In [4]:
DATA_DIR = "/kaggle/input/competitions/new-york-city-taxi-fare-prediction"

In [5]:
import os

DATA_DIR = "/kaggle/input/competitions/new-york-city-taxi-fare-prediction"

for file in os.listdir(DATA_DIR):
    path = os.path.join(DATA_DIR, file)

    if os.path.isfile(path):
        size_gb = os.path.getsize(path) / (1024 ** 3)
        print(f"{file:30s} {size_gb:.3f} GB")

sample_submission.csv          0.000 GB
GCP-Coupons-Instructions.rtf   0.000 GB
train.csv                      5.306 GB
test.csv                       0.001 GB


In [6]:
import pandas as pd

TRAIN_PATH = "/kaggle/input/competitions/new-york-city-taxi-fare-prediction/train.csv"

sample = pd.read_csv(TRAIN_PATH, nrows=5)

print("Shape of sampled data:", sample.shape)
print("\nColumns:")
print(sample.columns.tolist())

print("\nData types:")
print(sample.dtypes)

print("\nFirst 5 rows:")
display(sample)

Shape of sampled data: (5, 8)

Columns:
['key', 'fare_amount', 'pickup_datetime', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'passenger_count']

Data types:
key                   object
fare_amount          float64
pickup_datetime       object
pickup_longitude     float64
pickup_latitude      float64
dropoff_longitude    float64
dropoff_latitude     float64
passenger_count        int64
dtype: object

First 5 rows:


,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,2009-06-15 17:26:21.0000001,4.5,2009-06-15 17:26:21 UTC,-73.844311,40.721319,-73.841610,40.712278,1
1,2010-01-05 16:52:16.0000002,16.9,2010-01-05 16:52:16 UTC,-74.016048,40.711303,-73.979268,40.782004,1
2,2011-08-18 00:35:00.00000049,5.7,2011-08-18 00:35:00 UTC,-73.982738,40.761270,-73.991242,40.750562,2
3,2012-04-21 04:30:42.0000001,7.7,2012-04-21 04:30:42 UTC,-73.987130,40.733143,-73.991567,40.758092,1
4,2010-03-09 07:51:00.000000135,5.3,2010-03-09 07:51:00 UTC,-73.968095,40.768008,-73.956655,40.783762,1


In [7]:
row_count = 0

for chunk in pd.read_csv(TRAIN_PATH, chunksize=1_000_000):
    row_count += len(chunk)
    print(f"Processed: {row_count:,} rows")

print(f"\nTotal rows: {row_count:,}")

Processed: 1,000,000 rows
Processed: 2,000,000 rows
Processed: 3,000,000 rows
Processed: 4,000,000 rows
Processed: 5,000,000 rows
Processed: 6,000,000 rows
Processed: 7,000,000 rows
Processed: 8,000,000 rows
Processed: 9,000,000 rows
Processed: 10,000,000 rows
Processed: 11,000,000 rows
Processed: 12,000,000 rows
Processed: 13,000,000 rows
Processed: 14,000,000 rows
Processed: 15,000,000 rows
Processed: 16,000,000 rows
Processed: 17,000,000 rows
Processed: 18,000,000 rows
Processed: 19,000,000 rows
Processed: 20,000,000 rows
Processed: 21,000,000 rows
Processed: 22,000,000 rows
Processed: 23,000,000 rows
Processed: 24,000,000 rows
Processed: 25,000,000 rows
Processed: 26,000,000 rows
Processed: 27,000,000 rows
Processed: 28,000,000 rows
Processed: 29,000,000 rows
Processed: 30,000,000 rows
Processed: 31,000,000 rows
Processed: 32,000,000 rows
Processed: 33,000,000 rows
Processed: 34,000,000 rows
Processed: 35,000,000 rows
Processed: 36,000,000 rows
Processed: 37,000,000 rows
Processed:

In [8]:
sample_df = pd.read_csv(
    TRAIN_PATH,
    nrows=100_000
)

print("Shape:", sample_df.shape)

print("\nMissing values:")
print(sample_df.isna().sum())

print("\nData types:")
print(sample_df.dtypes)

print("\nNumerical summary:")
display(sample_df.describe())

Shape: (100000, 8)

Missing values:
key                  0
fare_amount          0
pickup_datetime      0
pickup_longitude     0
pickup_latitude      0
dropoff_longitude    0
dropoff_latitude     0
passenger_count      0
dtype: int64

Data types:
key                   object
fare_amount          float64
pickup_datetime       object
pickup_longitude     float64
pickup_latitude      float64
dropoff_longitude    float64
dropoff_latitude     float64
passenger_count        int64
dtype: object

Numerical summary:


,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,11.354652,-72.494682,39.914481,-72.490967,39.919053,1.673820
std,9.716777,10.693934,6.225686,10.471386,6.213427,1.300171
min,-44.900000,-736.550000,-74.007670,-84.654241,-74.006377,0.000000
25%,6.000000,-73.992041,40.734996,-73.991215,40.734182,1.000000
50%,8.500000,-73.981789,40.752765,-73.980000,40.753243,1.000000
75%,12.500000,-73.966982,40.767258,-73.963433,40.768166,2.000000
max,200.000000,40.787575,401.083332,40.851027,404.616667,6.000000


In [9]:
checks = {
    "negative_fare": sample_df["fare_amount"] <= 0,

    "invalid_pickup_longitude": ~sample_df["pickup_longitude"].between(-75, -72),

    "invalid_pickup_latitude": ~sample_df["pickup_latitude"].between(40, 42),

    "invalid_dropoff_longitude": ~sample_df["dropoff_longitude"].between(-75, -72),

    "invalid_dropoff_latitude": ~sample_df["dropoff_latitude"].between(40, 42),

    "invalid_passenger_count": sample_df["passenger_count"] <= 0,
}

for name, condition in checks.items():
    print(f"{name:30s}: {condition.sum():,} rows ({condition.mean()*100:.2f}%)")

negative_fare                 : 12 rows (0.01%)
invalid_pickup_longitude      : 1,995 rows (1.99%)
invalid_pickup_latitude       : 1,999 rows (2.00%)
invalid_dropoff_longitude     : 1,983 rows (1.98%)
invalid_dropoff_latitude      : 1,983 rows (1.98%)
invalid_passenger_count       : 366 rows (0.37%)


In [10]:
invalid_mask = (
    checks["negative_fare"]
    | checks["invalid_pickup_longitude"]
    | checks["invalid_pickup_latitude"]
    | checks["invalid_dropoff_longitude"]
    | checks["invalid_dropoff_latitude"]
    | checks["invalid_passenger_count"]
)

print("Total invalid rows:", invalid_mask.sum())
print("Percentage of sample:", f"{invalid_mask.mean()*100:.2f}%")

Total invalid rows: 2479
Percentage of sample: 2.48%


In [11]:
invalid_rows = sample_df[invalid_mask]

print("Invalid rows shape:", invalid_rows.shape)

display(invalid_rows.head(20))

print("\nSummary of invalid rows:")
display(invalid_rows.describe())

Invalid rows shape: (2479, 8)


,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
11,2012-12-24 11:24:00.00000098,5.5,2012-12-24 11:24:00 UTC,0.000000,0.000000,0.000000,0.000000,3
15,2013-11-23 12:57:00.000000190,5.0,2013-11-23 12:57:00 UTC,0.000000,0.000000,0.000000,0.000000,1
26,2011-02-07 20:01:00.000000114,6.5,2011-02-07 20:01:00 UTC,0.000000,0.000000,0.000000,0.000000,1
124,2013-01-17 17:22:00.00000043,8.0,2013-01-17 17:22:00 UTC,0.000000,0.000000,0.000000,0.000000,2
192,2010-09-05 17:08:00.00000092,3.7,2010-09-05 17:08:00 UTC,0.000000,0.000000,0.000000,0.000000,5
233,2011-07-24 01:14:35.0000002,8.5,2011-07-24 01:14:35 UTC,0.000000,0.000000,0.000000,0.000000,2
273,2009-10-30 18:13:00.00000021,8.1,2009-10-30 18:13:00 UTC,0.000000,0.000000,0.000000,0.000000,4
314,2015-06-02 23:16:15.00000012,34.0,2015-06-02 23:16:15 UTC,-73.974899,40.751095,-73.908546,40.881878,0
357,2013-07-04 16:41:27.0000002,8.5,2013-07-04 16:41:27 UTC,0.000000,0.000000,0.000000,0.000000,1
376,2014-05-29 05:57:22.0000001,2.5,2014-05-29 05:57:22 UTC,0.000000,0.000000,0.000000,0.000000,1



Summary of invalid rows:


,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,2479.000000,2479.000000,2479.000000,2479.000000,2479.000000,2479.000000
mean,11.444937,-14.257541,7.000855,-14.149769,7.169498,1.411860
std,10.495099,33.700587,21.278738,30.548428,21.393349,1.334719
min,-44.900000,-736.550000,-74.007670,-84.654241,-74.006377,0.000000
25%,5.700000,0.000000,0.000000,0.000000,0.000000,1.000000
50%,8.100000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,12.900000,0.000000,0.000000,0.000000,0.000000,1.000000
max,128.830000,40.787575,401.083332,40.851027,404.616667,6.000000


In [12]:
# ============================================================
# STEP 9 — SCAN ALL 55.4M ROWS WITHOUT LOADING THEM AT ONCE
# ============================================================
# We process 1 million rows at a time.
# This gives us statistics for the ENTIRE dataset while keeping
# RAM usage manageable.
#
# Important:
# We are ONLY counting problems here.
# We are NOT deleting or modifying any data yet.
# ============================================================

import pandas as pd

total_rows = 0

invalid_counts = {
    "negative_fare": 0,
    "invalid_pickup_longitude": 0,
    "invalid_pickup_latitude": 0,
    "invalid_dropoff_longitude": 0,
    "invalid_dropoff_latitude": 0,
    "invalid_passenger_count": 0,
    "any_invalid": 0
}

for chunk in pd.read_csv(TRAIN_PATH, chunksize=1_000_000):

    total_rows += len(chunk)

    negative_fare = chunk["fare_amount"] <= 0

    invalid_pickup_longitude = ~chunk["pickup_longitude"].between(-75, -72)
    invalid_pickup_latitude = ~chunk["pickup_latitude"].between(40, 42)

    invalid_dropoff_longitude = ~chunk["dropoff_longitude"].between(-75, -72)
    invalid_dropoff_latitude = ~chunk["dropoff_latitude"].between(40, 42)

    invalid_passenger_count = chunk["passenger_count"] <= 0

    any_invalid = (
        negative_fare
        | invalid_pickup_longitude
        | invalid_pickup_latitude
        | invalid_dropoff_longitude
        | invalid_dropoff_latitude
        | invalid_passenger_count
    )

    invalid_counts["negative_fare"] += negative_fare.sum()
    invalid_counts["invalid_pickup_longitude"] += invalid_pickup_longitude.sum()
    invalid_counts["invalid_pickup_latitude"] += invalid_pickup_latitude.sum()
    invalid_counts["invalid_dropoff_longitude"] += invalid_dropoff_longitude.sum()
    invalid_counts["invalid_dropoff_latitude"] += invalid_dropoff_latitude.sum()
    invalid_counts["invalid_passenger_count"] += invalid_passenger_count.sum()
    invalid_counts["any_invalid"] += any_invalid.sum()

    print(f"Processed {total_rows:,} rows")

print("\n========== FULL DATASET RESULTS ==========")
print(f"Total rows: {total_rows:,}")

for name, count in invalid_counts.items():
    print(f"{name:30s}: {count:,} ({count / total_rows * 100:.2f}%)")

Processed 1,000,000 rows
Processed 2,000,000 rows
Processed 3,000,000 rows
Processed 4,000,000 rows
Processed 5,000,000 rows
Processed 6,000,000 rows
Processed 7,000,000 rows
Processed 8,000,000 rows
Processed 9,000,000 rows
Processed 10,000,000 rows
Processed 11,000,000 rows
Processed 12,000,000 rows
Processed 13,000,000 rows
Processed 14,000,000 rows
Processed 15,000,000 rows
Processed 16,000,000 rows
Processed 17,000,000 rows
Processed 18,000,000 rows
Processed 19,000,000 rows
Processed 20,000,000 rows
Processed 21,000,000 rows
Processed 22,000,000 rows
Processed 23,000,000 rows
Processed 24,000,000 rows
Processed 25,000,000 rows
Processed 26,000,000 rows
Processed 27,000,000 rows
Processed 28,000,000 rows
Processed 29,000,000 rows
Processed 30,000,000 rows
Processed 31,000,000 rows
Processed 32,000,000 rows
Processed 33,000,000 rows
Processed 34,000,000 rows
Processed 35,000,000 rows
Processed 36,000,000 rows
Processed 37,000,000 rows
Processed 38,000,000 rows
Processed 39,000,000 

In [13]:

zero_coordinate_rows = 0
total_rows = 0

for chunk in pd.read_csv(TRAIN_PATH, chunksize=1_000_000):

    total_rows += len(chunk)

    zero_coords = (
        (chunk["pickup_longitude"] == 0)
        & (chunk["pickup_latitude"] == 0)
        & (chunk["dropoff_longitude"] == 0)
        & (chunk["dropoff_latitude"] == 0)
    )

    zero_coordinate_rows += zero_coords.sum()

print(f"Rows with all coordinates = 0,0: {zero_coordinate_rows:,}")
print(f"Percentage: {zero_coordinate_rows / total_rows * 100:.2f}%")

Rows with all coordinates = 0,0: 1,003,352
Percentage: 1.81%


In [14]:
nonzero_invalid_coords = 0
total_rows = 0

for chunk in pd.read_csv(TRAIN_PATH, chunksize=1_000_000):

    total_rows += len(chunk)

    invalid_coords = (
        ~chunk["pickup_longitude"].between(-75, -72)
        | ~chunk["pickup_latitude"].between(40, 42)
        | ~chunk["dropoff_longitude"].between(-75, -72)
        | ~chunk["dropoff_latitude"].between(40, 42)
    )

    all_zero = (
        (chunk["pickup_longitude"] == 0)
        & (chunk["pickup_latitude"] == 0)
        & (chunk["dropoff_longitude"] == 0)
        & (chunk["dropoff_latitude"] == 0)
    )

    nonzero_invalid_coords += (invalid_coords & ~all_zero).sum()

print(f"Non-(0,0) invalid-coordinate rows: {nonzero_invalid_coords:,}")
print(f"Percentage: {nonzero_invalid_coords / total_rows * 100:.2f}%")

Non-(0,0) invalid-coordinate rows: 163,203
Percentage: 0.29%


In [15]:
import os
import gc
import numpy as np
import pandas as pd

PROCESSED_DIR = "/kaggle/working/taxi_processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

CHUNK_SIZE = 1_000_000

chunk_number = 0
total_input = 0
total_removed = 0


for chunk in pd.read_csv(TRAIN_PATH, chunksize=CHUNK_SIZE):

    chunk_number += 1
    total_input += len(chunk)

    # --------------------------------------------------------
    # 1. Remove clearly invalid records
    # --------------------------------------------------------
    # These rules were verified against the FULL dataset.
    #
    # Coordinates must fall within a reasonable NYC bounding
    # box. Passenger count must be positive and fare must be
    # positive.
    # --------------------------------------------------------

    valid_mask = (
        chunk["fare_amount"].gt(0)
        & chunk["pickup_longitude"].between(-75, -72)
        & chunk["pickup_latitude"].between(40, 42)
        & chunk["dropoff_longitude"].between(-75, -72)
        & chunk["dropoff_latitude"].between(40, 42)
        & chunk["passenger_count"].gt(0)
    )

    removed = (~valid_mask).sum()
    total_removed += removed

    chunk = chunk.loc[valid_mask].copy()

    # --------------------------------------------------------
    # 2. Parse datetime
    # --------------------------------------------------------
    # The raw datetime string is not directly useful to most
    # ML models. We convert it into numerical time features.
    # --------------------------------------------------------

    dt = pd.to_datetime(chunk["pickup_datetime"], errors="coerce")

    chunk["year"] = dt.dt.year.astype("int16")
    chunk["month"] = dt.dt.month.astype("int8")
    chunk["hour"] = dt.dt.hour.astype("int8")
    chunk["day_of_week"] = dt.dt.dayofweek.astype("int8")

    # --------------------------------------------------------
    # 3. Calculate geographic distance
    # --------------------------------------------------------
    # Latitude/longitude describe the trip endpoints.
    # Haversine distance gives us an approximate straight-line
    # distance between pickup and dropoff.
    # --------------------------------------------------------

    lat1 = np.radians(chunk["pickup_latitude"].astype("float32"))
    lat2 = np.radians(chunk["dropoff_latitude"].astype("float32"))

    dlat = lat2 - lat1

    lon1 = np.radians(chunk["pickup_longitude"].astype("float32"))
    lon2 = np.radians(chunk["dropoff_longitude"].astype("float32"))

    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    chunk["distance_km"] = (
        6371 * 2 * np.arcsin(np.sqrt(a))
    ).astype("float32")

    # --------------------------------------------------------
    # 4. Remove columns we no longer need
    # --------------------------------------------------------
    # 'key' is an identifier and the raw datetime has already
    # been converted into useful numerical features.
    # The original coordinates are also removed because their
    # information is represented by distance.
    # --------------------------------------------------------

    chunk.drop(
        columns=[
            "key",
            "pickup_datetime",
            "pickup_longitude",
            "pickup_latitude",
            "dropoff_longitude",
            "dropoff_latitude"
        ],
        inplace=True
    )

    # --------------------------------------------------------
    # 5. Downcast numerical columns
    # --------------------------------------------------------
    # Smaller dtypes reduce RAM usage and storage size.
    # float32 is sufficient for these features; passenger count
    # and extracted time features need only small integers.
    # --------------------------------------------------------

    chunk["fare_amount"] = chunk["fare_amount"].astype("float32")
    chunk["passenger_count"] = chunk["passenger_count"].astype("int8")

    # --------------------------------------------------------
    # 6. Save this processed chunk as Parquet
    # --------------------------------------------------------
    # We save separate Parquet files instead of concatenating
    # 55M rows into one giant DataFrame.
    # --------------------------------------------------------

    output_path = os.path.join(
        PROCESSED_DIR,
        f"part_{chunk_number:03d}.parquet"
    )

    chunk.to_parquet(output_path, index=False)

    print(
        f"Chunk {chunk_number:02d} | "
        f"Input: {total_input:,} | "
        f"Removed: {removed:,} | "
        f"Saved: {len(chunk):,}"
    )

    # Explicitly release memory before reading the next chunk.
    del chunk, dt, lat1, lat2, dlat, lon1, lon2, dlon, a
    gc.collect()


print("\n========== PROCESSING COMPLETE ==========")
print(f"Total input rows : {total_input:,}")
print(f"Total removed    : {total_removed:,}")
print(f"Output directory : {PROCESSED_DIR}")

Chunk 01 | Input: 1,000,000 | Removed: 24,322 | Saved: 975,678
Chunk 02 | Input: 2,000,000 | Removed: 24,430 | Saved: 975,570
Chunk 03 | Input: 3,000,000 | Removed: 24,837 | Saved: 975,163
Chunk 04 | Input: 4,000,000 | Removed: 24,705 | Saved: 975,295
Chunk 05 | Input: 5,000,000 | Removed: 24,562 | Saved: 975,438
Chunk 06 | Input: 6,000,000 | Removed: 24,417 | Saved: 975,583
Chunk 07 | Input: 7,000,000 | Removed: 24,529 | Saved: 975,471
Chunk 08 | Input: 8,000,000 | Removed: 24,663 | Saved: 975,337
Chunk 09 | Input: 9,000,000 | Removed: 24,769 | Saved: 975,231
Chunk 10 | Input: 10,000,000 | Removed: 24,786 | Saved: 975,214
Chunk 11 | Input: 11,000,000 | Removed: 24,428 | Saved: 975,572
Chunk 12 | Input: 12,000,000 | Removed: 24,391 | Saved: 975,609
Chunk 13 | Input: 13,000,000 | Removed: 24,667 | Saved: 975,333
Chunk 14 | Input: 14,000,000 | Removed: 24,403 | Saved: 975,597
Chunk 15 | Input: 15,000,000 | Removed: 24,275 | Saved: 975,725
Chunk 16 | Input: 16,000,000 | Removed: 24,765 | 

In [16]:
import os
import glob
import pandas as pd

parquet_files = sorted(
    glob.glob("/kaggle/working/taxi_processed/*.parquet")
)

print("Number of Parquet files:", len(parquet_files))

# Read only the first processed chunk.
check_df = pd.read_parquet(parquet_files[0])

print("\nShape:", check_df.shape)

print("\nColumns:")
print(check_df.columns.tolist())

print("\nDtypes:")
print(check_df.dtypes)

print("\nMemory usage:")
print(f"{check_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\nFirst 5 rows:")
display(check_df.head())

# Calculate total size of the processed Parquet dataset.
total_size_gb = sum(
    os.path.getsize(file)
    for file in parquet_files
) / (1024 ** 3)

print(f"\nTotal Parquet size: {total_size_gb:.3f} GB")

Number of Parquet files: 56

Shape: (975678, 7)

Columns:
['fare_amount', 'passenger_count', 'year', 'month', 'hour', 'day_of_week', 'distance_km']

Dtypes:
fare_amount        float32
passenger_count       int8
year                 int16
month                 int8
hour                  int8
day_of_week           int8
distance_km        float32
dtype: object

Memory usage:
13.03 MB

First 5 rows:


,fare_amount,passenger_count,year,month,hour,day_of_week,distance_km
0,4.5,1,2009,6,17,0,1.031069
1,16.9,1,2010,1,16,1,8.449763
2,5.7,2,2011,8,0,3,1.389644
3,7.7,1,2012,4,4,5,2.799485
4,5.3,1,2010,3,7,1,1.998886



Total Parquet size: 0.414 GB


In [17]:
test_df = pd.read_csv(
    "/kaggle/input/competitions/new-york-city-taxi-fare-prediction/test.csv"
)

test_df["pickup_datetime"] = pd.to_datetime(
    test_df["pickup_datetime"],
    errors="coerce"
)

print("TEST PERIOD:")
print("Minimum:", test_df["pickup_datetime"].min())
print("Maximum:", test_df["pickup_datetime"].max())

print("\nTEST rows:", len(test_df))

# Count training rows by year using the already-processed
# Parquet files. We don't need to touch the original 5.3 GB CSV.
year_counts = {}

for file in parquet_files:
    part = pd.read_parquet(file, columns=["year"])

    counts = part["year"].value_counts()

    for year, count in counts.items():
        year_counts[year] = year_counts.get(year, 0) + count

print("\nTRAINING ROWS BY YEAR:")
for year, count in sorted(year_counts.items()):
    print(f"{year}: {count:,}")

TEST PERIOD:
Minimum: 2009-01-01 11:04:24+00:00
Maximum: 2015-06-30 20:03:50+00:00

TEST rows: 9914

TRAINING ROWS BY YEAR:
2009: 8,434,051
2010: 8,170,323
2011: 8,494,153
2012: 8,632,059
2013: 8,479,959
2014: 8,072,536
2015: 3,780,053


In [18]:
processed_rows = 0

for file in parquet_files:
    processed_rows += len(
        pd.read_parquet(file, columns=["fare_amount"])
    )

print(f"Processed rows: {processed_rows:,}")

Processed rows: 54,063,134


In [19]:
import os
import gc
import numpy as np
import pandas as pd

VALIDATION_FRACTION = 0.10
VALIDATION_SEED = 42

VALIDATION_DIR = "/kaggle/working/taxi_validation"
TRAIN_POOL_DIR = "/kaggle/working/taxi_train_pool"

os.makedirs(VALIDATION_DIR, exist_ok=True)
os.makedirs(TRAIN_POOL_DIR, exist_ok=True)

global_row_start = 0
validation_rows = 0
training_rows = 0

for file_number, file in enumerate(parquet_files, start=1):

    # --------------------------------------------------------
    # Read one Parquet chunk at a time.
    # --------------------------------------------------------
    part = pd.read_parquet(file)

    n = len(part)

    # --------------------------------------------------------
    # Create deterministic random numbers for these rows.
    #
    # The seed depends on the global row positions, so the same
    # row will always receive the same train/validation decision.
    # --------------------------------------------------------

    rng = np.random.default_rng(
        VALIDATION_SEED + file_number
    )

    validation_mask = (
        rng.random(n) < VALIDATION_FRACTION
    )

    validation_part = part.loc[validation_mask]
    training_part = part.loc[~validation_mask]

    # --------------------------------------------------------
    # Save validation and training-pool pieces separately.
    # --------------------------------------------------------

    validation_path = os.path.join(
        VALIDATION_DIR,
        f"validation_{file_number:03d}.parquet"
    )

    training_path = os.path.join(
        TRAIN_POOL_DIR,
        f"train_{file_number:03d}.parquet"
    )

    validation_part.to_parquet(
        validation_path,
        index=False
    )

    training_part.to_parquet(
        training_path,
        index=False
    )

    validation_rows += len(validation_part)
    training_rows += len(training_part)

    print(
        f"Part {file_number:02d} | "
        f"Train: {len(training_part):,} | "
        f"Validation: {len(validation_part):,}"
    )

    del part, validation_part, training_part
    gc.collect()


print("\n========== SPLIT COMPLETE ==========")
print(f"Training pool : {training_rows:,}")
print(f"Validation    : {validation_rows:,}")
print(f"Total         : {training_rows + validation_rows:,}")
print(
    f"Validation %  : "
    f"{validation_rows / (training_rows + validation_rows) * 100:.2f}%"
)

Part 01 | Train: 878,107 | Validation: 97,571
Part 02 | Train: 877,724 | Validation: 97,846
Part 03 | Train: 877,589 | Validation: 97,574
Part 04 | Train: 878,135 | Validation: 97,160
Part 05 | Train: 877,623 | Validation: 97,815
Part 06 | Train: 878,433 | Validation: 97,150
Part 07 | Train: 877,748 | Validation: 97,723
Part 08 | Train: 877,850 | Validation: 97,487
Part 09 | Train: 877,953 | Validation: 97,278
Part 10 | Train: 877,300 | Validation: 97,914
Part 11 | Train: 878,111 | Validation: 97,461
Part 12 | Train: 878,368 | Validation: 97,241
Part 13 | Train: 877,957 | Validation: 97,376
Part 14 | Train: 878,222 | Validation: 97,375
Part 15 | Train: 877,672 | Validation: 98,053
Part 16 | Train: 877,314 | Validation: 97,921
Part 17 | Train: 877,865 | Validation: 97,510
Part 18 | Train: 878,408 | Validation: 97,191
Part 19 | Train: 877,822 | Validation: 97,640
Part 20 | Train: 877,761 | Validation: 97,512
Part 21 | Train: 877,504 | Validation: 97,704
Part 22 | Train: 877,891 | Validat

In [20]:
import pandas as pd
import numpy as np
import glob

train_pool_files = sorted(
    glob.glob("/kaggle/working/taxi_train_pool/*.parquet")
)

validation_files = sorted(
    glob.glob("/kaggle/working/taxi_validation/*.parquet")
)

# ------------------------------------------------------------
# Collect a small sample from each side.
#
# We do NOT need millions of rows just to check whether the
# distributions look reasonable.
# ------------------------------------------------------------

train_samples = []
validation_samples = []

for train_file, val_file in zip(train_pool_files, validation_files):

    train_part = pd.read_parquet(
        train_file,
        columns=[
            "fare_amount",
            "distance_km",
            "year",
            "passenger_count"
        ]
    )

    val_part = pd.read_parquet(
        val_file,
        columns=[
            "fare_amount",
            "distance_km",
            "year",
            "passenger_count"
        ]
    )

    # Take a small random sample from each chunk.
    train_samples.append(
        train_part.sample(
            n=min(5_000, len(train_part)),
            random_state=42
        )
    )

    validation_samples.append(
        val_part.sample(
            n=min(5_000, len(val_part)),
            random_state=42
        )
    )


train_check = pd.concat(
    train_samples,
    ignore_index=True
)

validation_check = pd.concat(
    validation_samples,
    ignore_index=True
)

# ------------------------------------------------------------
# Compare numerical distributions.
# ------------------------------------------------------------

print("========== TRAIN vs VALIDATION ==========\n")

comparison = pd.DataFrame({
    "Train": train_check[
        ["fare_amount", "distance_km", "passenger_count"]
    ].mean(),

    "Validation": validation_check[
        ["fare_amount", "distance_km", "passenger_count"]
    ].mean()
})

print(comparison)

# ------------------------------------------------------------
# Compare year distribution.
# ------------------------------------------------------------

print("\n========== YEAR DISTRIBUTION ==========\n")

train_year_pct = (
    train_check["year"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

val_year_pct = (
    validation_check["year"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

year_comparison = pd.DataFrame({
    "Train %": train_year_pct,
    "Validation %": val_year_pct
}).fillna(0)

print(year_comparison.round(2))

# ------------------------------------------------------------
# Compare target quantiles.
#
# This is particularly important because fare_amount is our
# regression target.
# ------------------------------------------------------------

print("\n========== FARE QUANTILES ==========\n")

quantiles = [0.01, 0.25, 0.50, 0.75, 0.99]

fare_comparison = pd.DataFrame({
    "Train": train_check["fare_amount"].quantile(quantiles),
    "Validation": validation_check["fare_amount"].quantile(quantiles)
})

print(fare_comparison)

print("\n========== CHECK COMPLETE ==========")

========== TRAIN vs VALIDATION ==========

                     Train  Validation
fare_amount      11.336082   11.319543
distance_km       3.330868    3.320356
passenger_count   1.687696    1.692918

========== YEAR DISTRIBUTION ==========

      Train %  Validation %
year                       
2009    15.56         15.66
2010    15.17         15.06
2011    15.75         15.74
2012    15.94         15.92
2013    15.69         15.70
2014    14.92         14.87
2015     6.96          7.05

========== FARE QUANTILES ==========

      Train  Validation
0.01    3.3    3.300000
0.25    6.0    6.000000
0.50    8.5    8.500000
0.75   12.5   12.500000
0.99   52.0   52.830002

========== CHECK COMPLETE ==========


In [21]:
import os
import glob
import gc
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

TARGET_TRAIN_ROWS = 1_000_000
RANDOM_STATE = 42

TRAIN_POOL_DIR = "/kaggle/working/taxi_train_pool"

train_pool_files = sorted(
    glob.glob(os.path.join(TRAIN_POOL_DIR, "*.parquet"))
)

# These are the features currently available in our processed
# Parquet files.
FEATURES = [
    "passenger_count",
    "year",
    "month",
    "hour",
    "day_of_week",
    "distance_km"
]

TARGET = "fare_amount"

file_sizes = []

for file in train_pool_files:
    # Reading only the target column gives us the row count
    # while avoiding unnecessary columns.
    n_rows = len(
        pd.read_parquet(
            file,
            columns=[TARGET]
        )
    )

    file_sizes.append(n_rows)

total_training_rows = sum(file_sizes)

print(f"Training pool rows: {total_training_rows:,}")
print(f"Target sample size: {TARGET_TRAIN_ROWS:,}")

assert total_training_rows == 48_655_661, (
    "Training-pool row count does not match our fixed split."
)

raw_allocations = (
    np.array(file_sizes) / total_training_rows
) * TARGET_TRAIN_ROWS

sample_sizes = np.floor(raw_allocations).astype(int)

# Distribute the remaining rows caused by rounding.
remaining = TARGET_TRAIN_ROWS - sample_sizes.sum()

if remaining > 0:
    fractional_parts = raw_allocations - sample_sizes

    # Give the remaining rows to files with the largest
    # fractional remainders.
    largest_remainders = np.argsort(
        fractional_parts
    )[::-1][:remaining]

    sample_sizes[largest_remainders] += 1

assert sample_sizes.sum() == TARGET_TRAIN_ROWS

# ------------------------------------------------------------
# Sample each Parquet file.
# ------------------------------------------------------------

sample_parts = []

for i, (file, n_sample) in enumerate(
    zip(train_pool_files, sample_sizes)
):

    if n_sample == 0:
        continue

    # Read only the columns required for modeling.
    part = pd.read_parquet(
        file,
        columns=FEATURES + [TARGET]
    )

    # Use a deterministic seed for every file.
    # This makes the 1M sample reproducible.
    sampled_part = part.sample(
        n=n_sample,
        random_state=RANDOM_STATE + i
    )

    sample_parts.append(sampled_part)

    print(
        f"Part {i + 1:02d}: "
        f"{n_sample:,} sampled rows"
    )

    del part, sampled_part
    gc.collect()

train_1m = pd.concat(
    sample_parts,
    ignore_index=True
)

train_1m = train_1m.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)


assert len(train_1m) == TARGET_TRAIN_ROWS
assert train_1m[TARGET].notna().all()

print("\n========== 1M SAMPLE READY ==========")
print(f"Shape: {train_1m.shape}")
print(f"Rows : {len(train_1m):,}")
print(f"Memory: {train_1m.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

display(train_1m.head())

Training pool rows: 48,655,661
Target sample size: 1,000,000
Part 01: 18,047 sampled rows
Part 02: 18,040 sampled rows
Part 03: 18,037 sampled rows
Part 04: 18,048 sampled rows
Part 05: 18,037 sampled rows
Part 06: 18,054 sampled rows
Part 07: 18,040 sampled rows
Part 08: 18,042 sampled rows
Part 09: 18,044 sampled rows
Part 10: 18,031 sampled rows
Part 11: 18,047 sampled rows
Part 12: 18,053 sampled rows
Part 13: 18,044 sampled rows
Part 14: 18,050 sampled rows
Part 15: 18,038 sampled rows
Part 16: 18,031 sampled rows
Part 17: 18,042 sampled rows
Part 18: 18,054 sampled rows
Part 19: 18,042 sampled rows
Part 20: 18,040 sampled rows
Part 21: 18,035 sampled rows
Part 22: 18,043 sampled rows
Part 23: 18,055 sampled rows
Part 24: 18,041 sampled rows
Part 25: 18,033 sampled rows
Part 26: 18,043 sampled rows
Part 27: 18,036 sampled rows
Part 28: 18,045 sampled rows
Part 29: 18,032 sampled rows
Part 30: 18,045 sampled rows
Part 31: 18,037 sampled rows
Part 32: 18,048 sampled rows
Part 33: 18

,passenger_count,year,month,hour,day_of_week,distance_km,fare_amount
0,1,2009,8,19,4,1.729130,6.5
1,1,2011,11,10,0,0.715955,4.9
2,2,2015,1,20,5,3.654863,10.0
3,1,2011,3,4,6,1.912334,6.1
4,2,2013,9,18,5,5.252647,24.0


In [22]:
import os
import glob
import gc
import time

import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import mean_squared_error

FEATURES = [
    "passenger_count",
    "year",
    "month",
    "hour",
    "day_of_week",
    "distance_km"
]

TARGET = "fare_amount"

VALIDATION_DIR = "/kaggle/working/taxi_validation"

validation_files = sorted(
    glob.glob(
        os.path.join(
            VALIDATION_DIR,
            "*.parquet"
        )
    )
)


print("Loading fixed validation set...")

validation_parts = []

for file in validation_files:

    part = pd.read_parquet(
        file,
        columns=FEATURES + [TARGET]
    )

    validation_parts.append(part)

validation_df = pd.concat(
    validation_parts,
    ignore_index=True
)

del validation_parts
gc.collect()

print(
    f"Validation shape: {validation_df.shape}"
)


X_train_1m = train_1m[FEATURES]
y_train_1m = train_1m[TARGET]

X_val = validation_df[FEATURES]
y_val = validation_df[TARGET]


dtrain = xgb.DMatrix(
    X_train_1m,
    label=y_train_1m
)

dval = xgb.DMatrix(
    X_val,
    label=y_val
)

params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",

    # Tree complexity
    "max_depth": 8,
    "min_child_weight": 10,

    # Learning
    "eta": 0.10,

    # Randomization / regularization
    "subsample": 0.8,
    "colsample_bytree": 0.8,

    # Parallel CPU training
    "tree_method": "hist",
    "nthread": -1,

    "seed": 42
}

print("\n========== TRAINING XGBOOST ==========")

start_time = time.time()

xgb_1m_model = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=500,

    evals=[
        (dtrain, "train"),
        (dval, "validation")
    ],

    early_stopping_rounds=30,
    verbose_eval=25
)

training_time = time.time() - start_time


print("\n========== EVALUATION ==========")

val_predictions = xgb_1m_model.predict(
    dval
)

rmse_1m = np.sqrt(
    mean_squared_error(
        y_val,
        val_predictions
    )
)

print(f"1M Training RMSE : {rmse_1m:.6f}")
print(
    f"Best boosting round: "
    f"{xgb_1m_model.best_iteration}"
)
print(
    f"Training time: "
    f"{training_time / 60:.2f} minutes"
)

print("\n========== BASELINE COMPLETE ==========")



Loading fixed validation set...
Validation shape: (5407473, 7)

========== TRAINING XGBOOST ==========
[0]	train-rmse:8.96205	validation-rmse:33.78560
[25]	train-rmse:4.89731	validation-rmse:32.94387
[50]	train-rmse:4.51127	validation-rmse:32.89196
[75]	train-rmse:4.47469	validation-rmse:32.88995
[100]	train-rmse:4.44807	validation-rmse:32.88967
[120]	train-rmse:4.42929	validation-rmse:32.89022

========== EVALUATION ==========
1M Training RMSE : 32.890219
Best boosting round: 90
Training time: 0.47 minutes

========== BASELINE COMPLETE ==========


In [23]:
import time
import gc
import lightgbm as lgb
import numpy as np
from sklearn.metrics import mean_squared_error

FEATURES = [
    "passenger_count",
    "year",
    "month",
    "hour",
    "day_of_week",
    "distance_km"
]

TARGET = "fare_amount"

# ------------------------------------------------------------
# Prepare training data
# ------------------------------------------------------------

X_train_1m_lgb = train_1m[FEATURES]
y_train_1m_lgb = train_1m[TARGET]

# We already loaded the fixed validation set earlier.
X_val_lgb = validation_df[FEATURES]
y_val_lgb = validation_df[TARGET]

# ------------------------------------------------------------
# LightGBM datasets
# ------------------------------------------------------------

lgb_train = lgb.Dataset(
    X_train_1m_lgb,
    label=y_train_1m_lgb,
    free_raw_data=False
)

lgb_val = lgb.Dataset(
    X_val_lgb,
    label=y_val_lgb,
    reference=lgb_train,
    free_raw_data=False
)

# ------------------------------------------------------------
# Baseline parameters
# ------------------------------------------------------------

lgb_params = {
    "objective": "regression",
    "metric": "rmse",

    # Tree complexity
    "num_leaves": 64,
    "max_depth": -1,
    "min_data_in_leaf": 100,

    # Learning
    "learning_rate": 0.05,

    # Randomization / regularization
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,

    # Reproducibility / CPU
    "seed": 42,
    "verbosity": -1,
    "n_jobs": -1
}

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

print("========== TRAINING LIGHTGBM ==========")

start_time = time.time()

lgb_1m_model = lgb.train(
    params=lgb_params,
    train_set=lgb_train,
    num_boost_round=2000,

    valid_sets=[
        lgb_train,
        lgb_val
    ],

    valid_names=[
        "train",
        "validation"
    ],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            verbose=True
        )
    ]
)

training_time = time.time() - start_time

# ------------------------------------------------------------
# Predict on the SAME fixed validation set
# ------------------------------------------------------------

print("\n========== EVALUATION ==========")

lgb_predictions = lgb_1m_model.predict(
    X_val_lgb,
    num_iteration=lgb_1m_model.best_iteration
)

rmse_lgb_1m = np.sqrt(
    mean_squared_error(
        y_val_lgb,
        lgb_predictions
    )
)

print(f"1M LightGBM RMSE : {rmse_lgb_1m:.6f}")
print(
    f"Best boosting round: "
    f"{lgb_1m_model.best_iteration}"
)
print(
    f"Training time: "
    f"{training_time / 60:.2f} minutes"
)

# ------------------------------------------------------------
# Compare directly with our XGBoost baseline
# ------------------------------------------------------------

print("\n========== MODEL COMPARISON ==========")

print(f"XGBoost  : {rmse_1m:.6f}")
print(f"LightGBM : {rmse_lgb_1m:.6f}")

if rmse_lgb_1m < rmse_1m:
    print("LightGBM currently has the lower RMSE.")
else:
    print("XGBoost currently has the lower RMSE.")

print("\n========== LIGHTGBM BASELINE COMPLETE ==========")

# Free temporary LightGBM dataset objects.
del lgb_train, lgb_val
gc.collect()

========== TRAINING LIGHTGBM ==========
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[184]	train's rmse: 4.4722	validation's rmse: 32.8865

========== EVALUATION ==========
1M LightGBM RMSE : 32.886544
Best boosting round: 184
Training time: 0.71 minutes

========== MODEL COMPARISON ==========
XGBoost  : 32.890219
LightGBM : 32.886544
LightGBM currently has the lower RMSE.

========== LIGHTGBM BASELINE COMPLETE ==========


445

In [24]:
import time
import gc

import numpy as np

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_squared_error


# ------------------------------------------------------------
# 1. Prepare the SAME training and validation data
# ------------------------------------------------------------

FEATURES = [
    "passenger_count",
    "year",
    "month",
    "hour",
    "day_of_week",
    "distance_km"
]

TARGET = "fare_amount"

X_train_1m_et = train_1m[FEATURES]
y_train_1m_et = train_1m[TARGET]

X_val_et = validation_df[FEATURES]
y_val_et = validation_df[TARGET]


# ------------------------------------------------------------
# 2. Create ExtraTrees model
#
# ExtraTrees uses highly randomized decision trees.
# It can capture nonlinear relationships and interactions
# without requiring feature scaling.
#
# 100 trees is enough for this initial baseline.
# We will tune later only if the model is competitive.
# ------------------------------------------------------------

extra_trees_1m = ExtraTreesRegressor(
    n_estimators=100,
    max_depth=None,
    min_samples_leaf=5,
    max_features=1.0,

    # Use all available CPU cores.
    n_jobs=-1,

    random_state=42
)


# ------------------------------------------------------------
# 3. Train
# ------------------------------------------------------------

print("========== TRAINING EXTRA TREES ==========")

start_time = time.time()

extra_trees_1m.fit(
    X_train_1m_et,
    y_train_1m_et
)

training_time = time.time() - start_time


# ------------------------------------------------------------
# 4. Predict on the SAME fixed validation set
# ------------------------------------------------------------

print("\n========== EVALUATION ==========")

et_predictions = extra_trees_1m.predict(
    X_val_et
)

rmse_et_1m = np.sqrt(
    mean_squared_error(
        y_val_et,
        et_predictions
    )
)


# ------------------------------------------------------------
# 5. Compare all three candidates
# ------------------------------------------------------------

print(f"1M ExtraTrees RMSE : {rmse_et_1m:.6f}")
print(
    f"Training time: "
    f"{training_time / 60:.2f} minutes"
)

print("\n========== MODEL COMPARISON ==========")

results_1m = {
    "XGBoost": rmse_1m,
    "LightGBM": rmse_lgb_1m,
    "ExtraTrees": rmse_et_1m
}

for model_name, score in sorted(
    results_1m.items(),
    key=lambda x: x[1]
):
    print(f"{model_name:12s}: {score:.6f}")

print("\n========== EXTRA TREES BASELINE COMPLETE ==========")

========== TRAINING EXTRA TREES ==========

========== EVALUATION ==========
1M ExtraTrees RMSE : 32.882711
Training time: 1.89 minutes

========== MODEL COMPARISON ==========
ExtraTrees  : 32.882711
LightGBM    : 32.886544
XGBoost     : 32.890219

========== EXTRA TREES BASELINE COMPLETE ==========


In [25]:
residuals = pd.DataFrame({
    "XGBoost": y_val.to_numpy() - val_predictions,
    "LightGBM": y_val.to_numpy() - lgb_predictions,
    "ExtraTrees": y_val.to_numpy() - et_predictions
})

# ------------------------------------------------------------
# Calculate Pearson correlation between model errors.
# ------------------------------------------------------------

residual_correlation = residuals.corr()

print("========== RESIDUAL CORRELATION ==========\n")
print(residual_correlation.round(4))

# ------------------------------------------------------------
# Also calculate how often models disagree substantially.
# This gives us a simple view of prediction diversity.
# ------------------------------------------------------------

predictions = pd.DataFrame({
    "XGBoost": val_predictions,
    "LightGBM": lgb_predictions,
    "ExtraTrees": et_predictions
})

print("\n========== PREDICTION CORRELATION ==========\n")
print(predictions.corr().round(4))

print("\n========== INTERPRETATION ==========")
print(
    "Residual correlation closer to 1.0 means the models "
    "tend to make similar errors."
)
print(
    "Lower residual correlation means greater error diversity, "
    "which can make blending more useful."
)

========== RESIDUAL CORRELATION ==========

            XGBoost  LightGBM  ExtraTrees
XGBoost      1.0000    0.9998      0.9993
LightGBM     0.9998    1.0000      0.9994
ExtraTrees   0.9993    0.9994      1.0000

========== PREDICTION CORRELATION ==========

            XGBoost  LightGBM  ExtraTrees
XGBoost      1.0000    0.9977      0.9904
LightGBM     0.9977    1.0000      0.9909
ExtraTrees   0.9904    0.9909      1.0000

========== INTERPRETATION ==========
Residual correlation closer to 1.0 means the models tend to make similar errors.
Lower residual correlation means greater error diversity, which can make blending more useful.


In [26]:
import numpy as np
import pandas as pd


def create_taxi_features(df):
    """
    Create additional features using only information available
    at prediction time.

    The function does NOT use fare_amount, so there is no
    target leakage.
    """

    data = df.copy()

    # --------------------------------------------------------
    # 1. Rush-hour indicator
    #
    # NYC taxi demand/fare behavior can differ during commuting
    # hours. We encode the common morning/evening peak periods.
    # --------------------------------------------------------

    data["is_rush_hour"] = (
        data["hour"].isin([7, 8, 9, 16, 17, 18, 19])
    ).astype("int8")

    # --------------------------------------------------------
    # 2. Weekend indicator
    #
    # day_of_week:
    #     0 = Monday
    #     ...
    #     5 = Saturday
    #     6 = Sunday
    # --------------------------------------------------------

    data["is_weekend"] = (
        data["day_of_week"] >= 5
    ).astype("int8")

    # --------------------------------------------------------
    # 3. Late-night indicator
    # --------------------------------------------------------

    data["is_late_night"] = (
        data["hour"].isin([0, 1, 2, 3, 4, 5])
    ).astype("int8")

    # --------------------------------------------------------
    # 4. Business-hour indicator
    # --------------------------------------------------------

    data["is_business_hour"] = (
        data["hour"].between(9, 17)
    ).astype("int8")

    # --------------------------------------------------------
    # 5. Distance transformations
    #
    # log1p reduces the influence of very large distances.
    # This can help tree models distinguish short and long trips.
    # --------------------------------------------------------

    data["log_distance"] = np.log1p(
        data["distance_km"]
    ).astype("float32")

    # --------------------------------------------------------
    # 6. Distance × passenger interaction
    #
    # This allows the model to represent that the relationship
    # between distance and fare can vary with passenger count.
    # --------------------------------------------------------

    data["distance_per_passenger"] = (
        data["distance_km"] /
        data["passenger_count"].clip(lower=1)
    ).astype("float32")

    # --------------------------------------------------------
    # 7. Time interactions
    # --------------------------------------------------------

    data["hour_sin"] = np.sin(
        2 * np.pi * data["hour"] / 24
    ).astype("float32")

    data["hour_cos"] = np.cos(
        2 * np.pi * data["hour"] / 24
    ).astype("float32")

    # --------------------------------------------------------
    # 8. Month cyclic encoding
    #
    # December and January are adjacent in time even though
    # their integer values are far apart.
    # --------------------------------------------------------

    data["month_sin"] = np.sin(
        2 * np.pi * (data["month"] - 1) / 12
    ).astype("float32")

    data["month_cos"] = np.cos(
        2 * np.pi * (data["month"] - 1) / 12
    ).astype("float32")

    return data


# ------------------------------------------------------------
# Create the engineered 1M training dataset.
# ------------------------------------------------------------

train_1m_engineered = create_taxi_features(
    train_1m
)

# ------------------------------------------------------------
# Create the engineered validation dataset.
#
# IMPORTANT:
# The validation rows themselves are unchanged.
# We are only deriving new features from their existing
# predictor columns.
# ------------------------------------------------------------

validation_engineered = create_taxi_features(
    validation_df
)


# ------------------------------------------------------------
# Define final feature list.
# ------------------------------------------------------------

ENGINEERED_FEATURES = [
    "passenger_count",
    "year",
    "month",
    "hour",
    "day_of_week",
    "distance_km",

    "is_rush_hour",
    "is_weekend",
    "is_late_night",
    "is_business_hour",

    "log_distance",
    "distance_per_passenger",

    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos"
]

print("========== FEATURE ENGINEERING COMPLETE ==========")

print(
    f"Original features   : {len(FEATURES)}"
)

print(
    f"Engineered features : {len(ENGINEERED_FEATURES)}"
)

print(
    f"Training shape      : "
    f"{train_1m_engineered.shape}"
)

print(
    f"Validation shape    : "
    f"{validation_engineered.shape}"
)

print("\nNew features:")
print(
    [
        feature
        for feature in ENGINEERED_FEATURES
        if feature not in FEATURES
    ]
)

display(
    train_1m_engineered[
        ENGINEERED_FEATURES + [TARGET]
    ].head()
)

========== FEATURE ENGINEERING COMPLETE ==========
Original features   : 6
Engineered features : 16
Training shape      : (1000000, 17)
Validation shape    : (5407473, 17)

New features:
['is_rush_hour', 'is_weekend', 'is_late_night', 'is_business_hour', 'log_distance', 'distance_per_passenger', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos']


,passenger_count,year,month,hour,day_of_week,distance_km,is_rush_hour,is_weekend,is_late_night,is_business_hour,log_distance,distance_per_passenger,hour_sin,hour_cos,month_sin,month_cos,fare_amount
0,1,2009,8,19,4,1.729130,1,0,0,0,1.003983,1.729130,-0.965926,2.588190e-01,-0.500000,-0.866025,6.5
1,1,2011,11,10,0,0.715955,0,0,0,1,0.539970,0.715955,0.500000,-8.660254e-01,-0.866025,0.500000,4.9
2,2,2015,1,20,5,3.654863,0,1,0,0,1.537912,1.827431,-0.866025,5.000000e-01,0.000000,1.000000,10.0
3,1,2011,3,4,6,1.912334,0,1,1,0,1.068955,1.912334,0.866025,5.000000e-01,0.866025,0.500000,6.1
4,2,2013,9,18,5,5.252647,1,1,0,0,1.833005,2.626323,-1.000000,-1.836970e-16,-0.866025,-0.500000,24.0


In [27]:
import os
import gc
import numpy as np
import pandas as pd

RAW_PATH = "/kaggle/input/competitions/new-york-city-taxi-fare-prediction/train.csv"

SPATIAL_DIR = "/kaggle/working/taxi_spatial_processed"
os.makedirs(SPATIAL_DIR, exist_ok=True)

CHUNK_SIZE = 1_000_000

# ------------------------------------------------------------
# Haversine distance
# ------------------------------------------------------------
# Calculates great-circle distance between pickup and dropoff.
# This gives us a physically meaningful trip-distance feature.
# ------------------------------------------------------------

def haversine_km(lat1, lon1, lat2, lon2):

    R = 6371.0

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))


part_number = 0
total_input = 0
total_valid = 0

for chunk in pd.read_csv(RAW_PATH, chunksize=CHUNK_SIZE):

    total_input += len(chunk)

    # --------------------------------------------------------
    # Apply EXACTLY the same validity rules used previously.
    # This keeps the spatial dataset consistent with our
    # original cleaned dataset.
    # --------------------------------------------------------

    valid_mask = (
        (chunk["fare_amount"] > 0)
        & (chunk["pickup_longitude"].between(-75, -72))
        & (chunk["pickup_latitude"].between(40, 42))
        & (chunk["dropoff_longitude"].between(-75, -72))
        & (chunk["dropoff_latitude"].between(40, 42))
        & (chunk["passenger_count"] > 0)
    )

    chunk = chunk.loc[valid_mask].copy()

    if len(chunk) == 0:
        continue

    total_valid += len(chunk)

    # --------------------------------------------------------
    # Convert datetime into numerical time features.
    # These are available at prediction time.
    # --------------------------------------------------------

    dt = pd.to_datetime(chunk["pickup_datetime"], errors="coerce")

    chunk["year"] = dt.dt.year.astype("int16")
    chunk["month"] = dt.dt.month.astype("int8")
    chunk["hour"] = dt.dt.hour.astype("int8")
    chunk["day_of_week"] = dt.dt.dayofweek.astype("int8")

    # --------------------------------------------------------
    # Spatial features
    # --------------------------------------------------------

    chunk["distance_km"] = haversine_km(
        chunk["pickup_latitude"],
        chunk["pickup_longitude"],
        chunk["dropoff_latitude"],
        chunk["dropoff_longitude"]
    ).astype("float32")

    # Absolute coordinate differences capture spatial
    # displacement independently of the curved-earth distance.
    chunk["lat_difference"] = (
        chunk["dropoff_latitude"] - chunk["pickup_latitude"]
    ).abs().astype("float32")

    chunk["lon_difference"] = (
        chunk["dropoff_longitude"] - chunk["pickup_longitude"]
    ).abs().astype("float32")

    # --------------------------------------------------------
    # Keep coordinates because they contain location signal.
    #
    # We also remove columns that are not useful directly:
    # key and raw datetime.
    # --------------------------------------------------------

    keep_columns = [
        "fare_amount",
        "passenger_count",
        "year",
        "month",
        "hour",
        "day_of_week",
        "pickup_longitude",
        "pickup_latitude",
        "dropoff_longitude",
        "dropoff_latitude",
        "distance_km",
        "lat_difference",
        "lon_difference"
    ]

    chunk = chunk[keep_columns]

    # --------------------------------------------------------
    # Downcast numerical columns to reduce memory usage.
    # --------------------------------------------------------

    chunk["fare_amount"] = chunk["fare_amount"].astype("float32")
    chunk["passenger_count"] = chunk["passenger_count"].astype("int8")

    output_path = os.path.join(
        SPATIAL_DIR,
        f"part_{part_number:03d}.parquet"
    )

    chunk.to_parquet(output_path, index=False)

    part_number += 1

    print(
        f"Part {part_number:02d} | "
        f"rows: {len(chunk):,}"
    )

    del chunk
    gc.collect()


print("\n========== SPATIAL DATASET COMPLETE ==========")
print(f"Input rows       : {total_input:,}")
print(f"Valid rows       : {total_valid:,}")
print(f"Removed rows     : {total_input - total_valid:,}")
print(f"Parquet parts    : {part_number}")

Part 01 | rows: 975,678
Part 02 | rows: 975,570
Part 03 | rows: 975,163
Part 04 | rows: 975,295
Part 05 | rows: 975,438
Part 06 | rows: 975,583
Part 07 | rows: 975,471
Part 08 | rows: 975,337
Part 09 | rows: 975,231
Part 10 | rows: 975,214
Part 11 | rows: 975,572
Part 12 | rows: 975,609
Part 13 | rows: 975,333
Part 14 | rows: 975,597
Part 15 | rows: 975,725
Part 16 | rows: 975,235
Part 17 | rows: 975,375
Part 18 | rows: 975,599
Part 19 | rows: 975,462
Part 20 | rows: 975,273
Part 21 | rows: 975,208
Part 22 | rows: 975,514
Part 23 | rows: 975,802
Part 24 | rows: 975,290
Part 25 | rows: 975,332
Part 26 | rows: 975,071
Part 27 | rows: 975,350
Part 28 | rows: 975,664
Part 29 | rows: 975,404
Part 30 | rows: 975,428
Part 31 | rows: 975,269
Part 32 | rows: 975,411
Part 33 | rows: 975,414
Part 34 | rows: 975,397
Part 35 | rows: 975,699
Part 36 | rows: 975,393
Part 37 | rows: 975,633
Part 38 | rows: 975,431
Part 39 | rows: 975,241
Part 40 | rows: 975,570
Part 41 | rows: 975,381
Part 42 | rows: 

In [28]:
# ============================================================
# VERIFY THE NEW SPATIAL DATASET
# ============================================================

spatial_files = sorted([
    os.path.join(SPATIAL_DIR, f)
    for f in os.listdir(SPATIAL_DIR)
    if f.endswith(".parquet")
])

check = pd.read_parquet(spatial_files[0])

print("Shape:", check.shape)
print("\nColumns:")
print(check.columns.tolist())

print("\nDtypes:")
print(check.dtypes)

print("\nSample:")
display(check.head())

Shape: (975678, 13)

Columns:
['fare_amount', 'passenger_count', 'year', 'month', 'hour', 'day_of_week', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'distance_km', 'lat_difference', 'lon_difference']

Dtypes:
fare_amount          float32
passenger_count         int8
year                   int16
month                   int8
hour                    int8
day_of_week             int8
pickup_longitude     float64
pickup_latitude      float64
dropoff_longitude    float64
dropoff_latitude     float64
distance_km          float32
lat_difference       float32
lon_difference       float32
dtype: object

Sample:


,fare_amount,passenger_count,year,month,hour,day_of_week,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,distance_km,lat_difference,lon_difference
0,4.5,1,2009,6,17,0,-73.844311,40.721319,-73.841610,40.712278,1.030764,0.009041,0.002701
1,16.9,1,2010,1,16,1,-74.016048,40.711303,-73.979268,40.782004,8.450133,0.070701,0.036780
2,5.7,2,2011,8,0,3,-73.982738,40.761270,-73.991242,40.750562,1.389525,0.010708,0.008504
3,7.7,1,2012,4,4,5,-73.987130,40.733143,-73.991567,40.758092,2.799270,0.024949,0.004437
4,5.3,1,2010,3,7,1,-73.968095,40.768008,-73.956655,40.783762,1.999157,0.015754,0.011440


In [29]:
import os
import gc
import numpy as np
import pandas as pd

SPATIAL_DIR = "/kaggle/working/taxi_spatial_processed"

TRAIN_POOL_DIR = "/kaggle/working/taxi_spatial_train_pool"
VALIDATION_DIR = "/kaggle/working/taxi_spatial_validation"

os.makedirs(TRAIN_POOL_DIR, exist_ok=True)
os.makedirs(VALIDATION_DIR, exist_ok=True)

VALIDATION_FRACTION = 0.10
VALIDATION_SEED = 42

spatial_files = sorted([
    os.path.join(SPATIAL_DIR, f)
    for f in os.listdir(SPATIAL_DIR)
    if f.endswith(".parquet")
])

total_rows = 0
total_train = 0
total_validation = 0

# ------------------------------------------------------------
# Process every Parquet part independently.
#
# A fixed random seed derived from the part number makes the
# assignment reproducible: running this cell again produces
# the same train/validation membership.
# ------------------------------------------------------------

for part_number, file_path in enumerate(spatial_files):

    part = pd.read_parquet(file_path)

    rng = np.random.default_rng(
        VALIDATION_SEED + part_number
    )

    # Generate a random number for every row.
    # Rows below the validation threshold go to validation.
    random_values = rng.random(len(part))

    validation_mask = random_values < VALIDATION_FRACTION
    train_mask = ~validation_mask

    validation_part = part.loc[validation_mask].copy()
    train_part = part.loc[train_mask].copy()

    train_path = os.path.join(
        TRAIN_POOL_DIR,
        f"train_{part_number:03d}.parquet"
    )

    validation_path = os.path.join(
        VALIDATION_DIR,
        f"validation_{part_number:03d}.parquet"
    )

    train_part.to_parquet(train_path, index=False)
    validation_part.to_parquet(validation_path, index=False)

    total_rows += len(part)
    total_train += len(train_part)
    total_validation += len(validation_part)

    print(
        f"Part {part_number + 1:02d}/{len(spatial_files)} | "
        f"rows={len(part):,} | "
        f"train={len(train_part):,} | "
        f"validation={len(validation_part):,}"
    )

    del part, train_part, validation_part, random_values
    gc.collect()


print("\n========== FIXED SPATIAL SPLIT COMPLETE ==========")
print(f"Total rows      : {total_rows:,}")
print(f"Training pool   : {total_train:,}")
print(f"Validation      : {total_validation:,}")
print(f"Validation %    : {total_validation / total_rows * 100:.2f}%")

Part 01/56 | rows=975,678 | train=878,402 | validation=97,276
Part 02/56 | rows=975,570 | train=878,006 | validation=97,564
Part 03/56 | rows=975,163 | train=877,359 | validation=97,804
Part 04/56 | rows=975,295 | train=877,706 | validation=97,589
Part 05/56 | rows=975,438 | train=878,265 | validation=97,173
Part 06/56 | rows=975,583 | train=877,750 | validation=97,833
Part 07/56 | rows=975,471 | train=878,329 | validation=97,142
Part 08/56 | rows=975,337 | train=877,629 | validation=97,708
Part 09/56 | rows=975,231 | train=877,751 | validation=97,480
Part 10/56 | rows=975,214 | train=877,937 | validation=97,277
Part 11/56 | rows=975,572 | train=877,620 | validation=97,952
Part 12/56 | rows=975,609 | train=878,148 | validation=97,461
Part 13/56 | rows=975,333 | train=878,114 | validation=97,219
Part 14/56 | rows=975,597 | train=878,197 | validation=97,400
Part 15/56 | rows=975,725 | train=878,342 | validation=97,383
Part 16/56 | rows=975,235 | train=877,231 | validation=98,004
Part 17/

In [30]:
import os
import numpy as np
import pandas as pd

SPATIAL_TRAIN_POOL_DIR = "/kaggle/working/taxi_spatial_train_pool"

TARGET = "fare_amount"

SPATIAL_FEATURES = [
    "passenger_count",
    "year",
    "month",
    "hour",
    "day_of_week",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "distance_km",
    "lat_difference",
    "lon_difference"
]

TARGET_TRAIN_ROWS = 1_000_000
RANDOM_STATE = 42

train_pool_files = sorted([
    os.path.join(SPATIAL_TRAIN_POOL_DIR, f)
    for f in os.listdir(SPATIAL_TRAIN_POOL_DIR)
    if f.endswith(".parquet")
])

print(f"Training pool files: {len(train_pool_files)}")

# ------------------------------------------------------------
# First determine the number of rows in every training part.
#
# We read ONLY the target column here, avoiding unnecessary
# loading of all 12 features just to count rows.
# ------------------------------------------------------------

part_sizes = []

for file_path in train_pool_files:

    n_rows = len(
        pd.read_parquet(
            file_path,
            columns=[TARGET]
        )
    )

    part_sizes.append(n_rows)

total_pool_rows = sum(part_sizes)

print(f"Training pool rows: {total_pool_rows:,}")

assert total_pool_rows == 48_655_705, (
    f"Unexpected training pool size: {total_pool_rows:,}"
)

# ------------------------------------------------------------
# Allocate the 1M samples proportionally across Parquet parts.
#
# Example:
# If a part contains 2% of the training pool, approximately
# 2% of our 1M sample will come from that part.
#
# This avoids accidentally over-representing small parts.
# ------------------------------------------------------------

exact_allocations = (
    np.array(part_sizes, dtype=float)
    / total_pool_rows
    * TARGET_TRAIN_ROWS
)

sample_counts = np.floor(exact_allocations).astype(int)

# We may be short by a few rows because of flooring.
# Give the remaining rows to parts with the largest
# fractional remainders.
remaining = TARGET_TRAIN_ROWS - sample_counts.sum()

if remaining > 0:

    fractional_parts = exact_allocations - sample_counts

    largest_remainders = np.argsort(
        fractional_parts
    )[-remaining:]

    sample_counts[largest_remainders] += 1

assert sample_counts.sum() == TARGET_TRAIN_ROWS

print(f"Rows to sample: {sample_counts.sum():,}")

# ------------------------------------------------------------
# Read only the required features + target from each part.
# ------------------------------------------------------------

sample_parts = []

for i, (file_path, n_sample) in enumerate(
    zip(train_pool_files, sample_counts)
):

    if n_sample == 0:
        continue

    part = pd.read_parquet(
        file_path,
        columns=SPATIAL_FEATURES + [TARGET]
    )

    # Fixed random_state makes this experiment reproducible.
    sampled_part = part.sample(
        n=n_sample,
        random_state=RANDOM_STATE + i
    )

    sample_parts.append(sampled_part)

    print(
        f"Part {i + 1:02d}/{len(train_pool_files)} | "
        f"available={len(part):,} | "
        f"sampled={n_sample:,}"
    )

# ------------------------------------------------------------
# Combine the sampled pieces and shuffle once.
#
# The shuffle prevents the final dataframe from being ordered
# according to the original Parquet parts.
# ------------------------------------------------------------

train_1m_spatial = pd.concat(
    sample_parts,
    ignore_index=True
)

train_1m_spatial = train_1m_spatial.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

# ------------------------------------------------------------
# Final integrity checks.
# ------------------------------------------------------------

assert len(train_1m_spatial) == TARGET_TRAIN_ROWS

assert list(train_1m_spatial.columns) == (
    SPATIAL_FEATURES + [TARGET]
)

print("\n========== 1M SPATIAL SAMPLE COMPLETE ==========")
print(f"Shape : {train_1m_spatial.shape}")
print(f"Rows  : {len(train_1m_spatial):,}")
print(f"Memory: {train_1m_spatial.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

display(train_1m_spatial.head())

Training pool files: 56
Training pool rows: 48,655,705
Rows to sample: 1,000,000
Part 01/56 | available=878,402 | sampled=18,053
Part 02/56 | available=878,006 | sampled=18,045
Part 03/56 | available=877,359 | sampled=18,032
Part 04/56 | available=877,706 | sampled=18,039
Part 05/56 | available=878,265 | sampled=18,051
Part 06/56 | available=877,750 | sampled=18,040
Part 07/56 | available=878,329 | sampled=18,052
Part 08/56 | available=877,629 | sampled=18,038
Part 09/56 | available=877,751 | sampled=18,040
Part 10/56 | available=877,937 | sampled=18,044
Part 11/56 | available=877,620 | sampled=18,037
Part 12/56 | available=878,148 | sampled=18,048
Part 13/56 | available=878,114 | sampled=18,048
Part 14/56 | available=878,197 | sampled=18,049
Part 15/56 | available=878,342 | sampled=18,052
Part 16/56 | available=877,231 | sampled=18,029
Part 17/56 | available=877,439 | sampled=18,034
Part 18/56 | available=878,072 | sampled=18,047
Part 19/56 | available=878,289 | sampled=18,051
Part 20

,passenger_count,year,month,hour,day_of_week,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,distance_km,lat_difference,lon_difference,fare_amount
0,2,2013,7,9,4,-73.991157,40.721070,-74.003912,40.728450,1.352305,0.007380,0.012755,7.5
1,1,2009,4,1,4,-73.988663,40.722409,-73.977208,40.746802,2.878981,0.024393,0.011455,7.0
2,2,2013,12,0,5,-73.982220,40.742927,-73.984892,40.752837,1.124696,0.009910,0.002672,5.5
3,5,2011,6,20,0,-73.950887,40.783192,-73.984895,40.745028,5.119745,0.038164,0.034008,11.3
4,1,2012,5,21,3,-73.984195,40.740362,-73.924120,40.766318,5.825522,0.025956,0.060075,16.1


In [31]:
import os
import gc
import time
import numpy as np
import pandas as pd
import xgboost as xgb

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

SPATIAL_VALIDATION_DIR = "/kaggle/working/taxi_spatial_validation"

# ------------------------------------------------------------
# Feature definition
# ------------------------------------------------------------

TARGET = "fare_amount"

SPATIAL_FEATURES = [
    "passenger_count",
    "year",
    "month",
    "hour",
    "day_of_week",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "distance_km",
    "lat_difference",
    "lon_difference"
]

# ------------------------------------------------------------
# Load the frozen spatial validation set.
#
# This is the validation set created earlier from the spatial
# dataset. We use ALL ~5.4M validation rows so the comparison
# remains statistically stable.
# ------------------------------------------------------------

validation_files = sorted([
    os.path.join(SPATIAL_VALIDATION_DIR, f)
    for f in os.listdir(SPATIAL_VALIDATION_DIR)
    if f.endswith(".parquet")
])

validation_parts = []

for file_path in validation_files:

    part = pd.read_parquet(
        file_path,
        columns=SPATIAL_FEATURES + [TARGET]
    )

    validation_parts.append(part)

validation_spatial = pd.concat(
    validation_parts,
    ignore_index=True
)

del validation_parts
gc.collect()

print("Validation shape:", validation_spatial.shape)

assert len(validation_spatial) == 5_407_429

# ------------------------------------------------------------
# Prepare training and validation matrices.
# ------------------------------------------------------------

X_train_spatial = train_1m_spatial[SPATIAL_FEATURES]
y_train_spatial = train_1m_spatial[TARGET]

X_val_spatial = validation_spatial[SPATIAL_FEATURES]
y_val_spatial = validation_spatial[TARGET]

print("Training shape  :", X_train_spatial.shape)
print("Validation shape:", X_val_spatial.shape)

# ------------------------------------------------------------
# XGBoost DMatrix
#
# DMatrix is XGBoost's optimized data structure for training.
# ------------------------------------------------------------

dtrain_spatial = xgb.DMatrix(
    X_train_spatial,
    label=y_train_spatial
)

dval_spatial = xgb.DMatrix(
    X_val_spatial,
    label=y_val_spatial
)

# ------------------------------------------------------------
# SAME hyperparameters as the previous 1M XGBoost experiment.
#
# We intentionally do NOT tune anything here.
# This keeps the feature-engineering comparison controlled.
# ------------------------------------------------------------

xgb_params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",

    "max_depth": 8,
    "min_child_weight": 10,

    "eta": 0.10,

    "subsample": 0.8,
    "colsample_bytree": 0.8,

    "tree_method": "hist",

    "nthread": -1,
    "seed": 42
}

# ------------------------------------------------------------
# Train with early stopping.
#
# The validation set is monitored after each boosting round.
# If RMSE does not improve for 30 rounds, training stops.
# ------------------------------------------------------------

start_time = time.time()

xgb_spatial_model = xgb.train(
    params=xgb_params,
    dtrain=dtrain_spatial,
    num_boost_round=500,

    evals=[
        (dtrain_spatial, "train"),
        (dval_spatial, "validation")
    ],

    early_stopping_rounds=30,
    verbose_eval=25
)

training_time = (time.time() - start_time) / 60

# ------------------------------------------------------------
# Final validation prediction.
#
# IMPORTANT:
# XGBoost automatically uses the best iteration when the model
# was trained with early stopping.
# ------------------------------------------------------------

val_predictions_spatial_xgb = xgb_spatial_model.predict(
    dval_spatial
)

spatial_xgb_rmse = np.sqrt(
    np.mean(
        (y_val_spatial.to_numpy() - val_predictions_spatial_xgb) ** 2
    )
)

print("\n========== SPATIAL XGBOOST RESULT ==========")
print(f"Validation RMSE : {spatial_xgb_rmse:.6f}")
print(f"Best iteration  : {xgb_spatial_model.best_iteration}")
print(f"Training time   : {training_time:.2f} minutes")

print("\n========== COMPARISON ==========")
print("Previous XGBoost RMSE : 32.890219")
print(f"Spatial XGBoost RMSE  : {spatial_xgb_rmse:.6f}")
print(
    f"RMSE improvement      : "
    f"{32.890219 - spatial_xgb_rmse:.6f}"
)

Validation shape: (5407429, 13)
Training shape  : (1000000, 12)
Validation shape: (5407429, 12)
[0]	train-rmse:9.17057	validation-rmse:8.97790
[25]	train-rmse:4.41131	validation-rmse:4.07256
[50]	train-rmse:4.19311	validation-rmse:3.93199
[75]	train-rmse:4.07802	validation-rmse:3.90733
[100]	train-rmse:3.98513	validation-rmse:3.88830
[125]	train-rmse:3.92476	validation-rmse:3.87266
[150]	train-rmse:3.87014	validation-rmse:3.86606
[175]	train-rmse:3.82217	validation-rmse:3.86286
[200]	train-rmse:3.76445	validation-rmse:3.86296
[225]	train-rmse:3.71341	validation-rmse:3.86307

========== SPATIAL XGBOOST RESULT ==========
Validation RMSE : 3.863074
Best iteration  : 195
Training time   : 0.83 minutes

========== COMPARISON ==========
Previous XGBoost RMSE : 32.890219
Spatial XGBoost RMSE  : 3.863074
RMSE improvement      : 29.027147


In [32]:
import os
import numpy as np
import pandas as pd

OLD_DIR = "/kaggle/working/taxi_processed"
SPATIAL_DIR = "/kaggle/working/taxi_spatial_processed"

old_files = sorted([
    os.path.join(OLD_DIR, f)
    for f in os.listdir(OLD_DIR)
    if f.endswith(".parquet")
])

spatial_files = sorted([
    os.path.join(SPATIAL_DIR, f)
    for f in os.listdir(SPATIAL_DIR)
    if f.endswith(".parquet")
])

print("Old processed parts    :", len(old_files))
print("Spatial processed parts:", len(spatial_files))

assert len(old_files) == len(spatial_files) == 56


# ------------------------------------------------------------
# Haversine calculation used for an independent verification.
# ------------------------------------------------------------

def haversine_km_check(lat1, lon1, lat2, lon2):

    R = 6371.0

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))


# ------------------------------------------------------------
# Compare several corresponding parts.
#
# We don't need to scan 54M rows for this post-hoc check.
# A deterministic sample from several parts is sufficient to
# verify whether the implementations agree.
# ------------------------------------------------------------

parts_to_check = [0, 10, 20, 30, 40, 50, 55]

comparison_results = []

for part_idx in parts_to_check:

    old = pd.read_parquet(
        old_files[part_idx],
        columns=[
            "passenger_count",
            "year",
            "month",
            "hour",
            "day_of_week",
            "distance_km"
        ]
    )

    spatial = pd.read_parquet(
        spatial_files[part_idx],
        columns=[
            "passenger_count",
            "year",
            "month",
            "hour",
            "day_of_week",
            "pickup_longitude",
            "pickup_latitude",
            "dropoff_longitude",
            "dropoff_latitude",
            "distance_km"
        ]
    )

    # The rows should correspond because both datasets were
    # generated from the same cleaned source chunks.
    assert len(old) == len(spatial), (
        f"Row count mismatch in part {part_idx}"
    )

    # Select deterministic row positions.
    rng = np.random.default_rng(42 + part_idx)

    n_check = min(1000, len(spatial))

    indices = rng.choice(
        len(spatial),
        size=n_check,
        replace=False
    )

    old_sample = old.iloc[indices].reset_index(drop=True)
    spatial_sample = spatial.iloc[indices].reset_index(drop=True)

    # --------------------------------------------------------
    # Check that the non-spatial columns still correspond.
    # --------------------------------------------------------

    assert np.array_equal(
        old_sample["passenger_count"].to_numpy(),
        spatial_sample["passenger_count"].to_numpy()
    )

    assert np.array_equal(
        old_sample["year"].to_numpy(),
        spatial_sample["year"].to_numpy()
    )

    assert np.array_equal(
        old_sample["month"].to_numpy(),
        spatial_sample["month"].to_numpy()
    )

    assert np.array_equal(
        old_sample["hour"].to_numpy(),
        spatial_sample["hour"].to_numpy()
    )

    assert np.array_equal(
        old_sample["day_of_week"].to_numpy(),
        spatial_sample["day_of_week"].to_numpy()
    )

    # --------------------------------------------------------
    # Compare old and new stored distance values.
    # --------------------------------------------------------

    distance_difference = np.abs(
        old_sample["distance_km"].to_numpy()
        - spatial_sample["distance_km"].to_numpy()
    )

    # --------------------------------------------------------
    # Independently recompute Haversine distance from the
    # spatial coordinates.
    # --------------------------------------------------------

    recomputed_distance = haversine_km_check(
        spatial_sample["pickup_latitude"].to_numpy(),
        spatial_sample["pickup_longitude"].to_numpy(),
        spatial_sample["dropoff_latitude"].to_numpy(),
        spatial_sample["dropoff_longitude"].to_numpy()
    )

    recomputation_difference = np.abs(
        recomputed_distance
        - spatial_sample["distance_km"].to_numpy()
    )

    comparison_results.append({
        "part": part_idx + 1,
        "rows_checked": n_check,
        "max_old_vs_new_distance_diff":
            distance_difference.max(),
        "mean_old_vs_new_distance_diff":
            distance_difference.mean(),
        "max_recomputed_distance_diff":
            recomputation_difference.max(),
        "mean_recomputed_distance_diff":
            recomputation_difference.mean()
    })

comparison_df = pd.DataFrame(comparison_results)

display(comparison_df)

print("\n========== DISTANCE CONSISTENCY CHECK ==========")

print(
    "Maximum old-vs-new distance difference:",
    comparison_df["max_old_vs_new_distance_diff"].max()
)

print(
    "Maximum independent recomputation difference:",
    comparison_df["max_recomputed_distance_diff"].max()
)


Old processed parts    : 56
Spatial processed parts: 56


,part,rows_checked,max_old_vs_new_distance_diff,mean_old_vs_new_distance_diff,max_recomputed_distance_diff,mean_recomputed_distance_diff
0,1,1000,0.000872,0.000218,1.538682e-06,7.554931e-08
1,11,1000,0.000890,0.000223,8.473645e-07,6.936075e-08
2,21,1000,0.000996,0.000223,8.155639e-07,6.617016e-08
3,31,1000,0.000974,0.000216,2.509524e-06,6.720210e-08
4,41,1000,0.000916,0.000233,2.878474e-06,7.764898e-08
5,51,1000,0.001047,0.000218,9.426227e-07,7.095021e-08
6,56,1000,0.000930,0.000211,9.483228e-07,6.781975e-08



========== DISTANCE CONSISTENCY CHECK ==========
Maximum old-vs-new distance difference: 0.0010471343994140625
Maximum independent recomputation difference: 2.8784742482912407e-06


In [33]:
import os
import gc
import time
import numpy as np
import pandas as pd
import lightgbm as lgb

SPATIAL_VALIDATION_DIR = "/kaggle/working/taxi_spatial_validation"

TARGET = "fare_amount"

SPATIAL_FEATURES = [
    "passenger_count",
    "year",
    "month",
    "hour",
    "day_of_week",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "distance_km",
    "lat_difference",
    "lon_difference"
]

# ------------------------------------------------------------
# Load the SAME frozen validation set used by Spatial XGBoost.
# ------------------------------------------------------------

validation_files = sorted([
    os.path.join(SPATIAL_VALIDATION_DIR, f)
    for f in os.listdir(SPATIAL_VALIDATION_DIR)
    if f.endswith(".parquet")
])

validation_parts = []

for file_path in validation_files:

    part = pd.read_parquet(
        file_path,
        columns=SPATIAL_FEATURES + [TARGET]
    )

    validation_parts.append(part)

validation_spatial = pd.concat(
    validation_parts,
    ignore_index=True
)

del validation_parts
gc.collect()

assert len(validation_spatial) == 5_407_429

print("Validation shape:", validation_spatial.shape)

# ------------------------------------------------------------
# Prepare train / validation matrices.
# ------------------------------------------------------------

X_train_spatial = train_1m_spatial[SPATIAL_FEATURES]
y_train_spatial = train_1m_spatial[TARGET]

X_val_spatial = validation_spatial[SPATIAL_FEATURES]
y_val_spatial = validation_spatial[TARGET]

print("Training shape  :", X_train_spatial.shape)
print("Validation shape:", X_val_spatial.shape)

# ------------------------------------------------------------
# SAME LightGBM parameters used in our previous 1M baseline.
#
# We deliberately do not tune them here.
# ------------------------------------------------------------

lgb_params = {
    "objective": "regression",
    "metric": "rmse",

    "num_leaves": 64,
    "max_depth": -1,
    "min_data_in_leaf": 100,

    "learning_rate": 0.05,

    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,

    "seed": 42,
    "verbosity": -1,
    "n_jobs": -1
}

# ------------------------------------------------------------
# LightGBM dataset objects.
# ------------------------------------------------------------

lgb_train = lgb.Dataset(
    X_train_spatial,
    label=y_train_spatial
)

lgb_validation = lgb.Dataset(
    X_val_spatial,
    label=y_val_spatial,
    reference=lgb_train
)

# ------------------------------------------------------------
# Train with the SAME early-stopping setup as before.
# ------------------------------------------------------------

start_time = time.time()

lgb_spatial_model = lgb.train(
    params=lgb_params,
    train_set=lgb_train,
    num_boost_round=2000,

    valid_sets=[
        lgb_train,
        lgb_validation
    ],

    valid_names=[
        "train",
        "validation"
    ],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            verbose=True
        )
    ]
)

training_time = (time.time() - start_time) / 60

# ------------------------------------------------------------
# Predict on the frozen validation set using the best iteration.
# ------------------------------------------------------------

val_predictions_spatial_lgb = lgb_spatial_model.predict(
    X_val_spatial,
    num_iteration=lgb_spatial_model.best_iteration
)

spatial_lgb_rmse = np.sqrt(
    np.mean(
        (
            y_val_spatial.to_numpy()
            - val_predictions_spatial_lgb
        ) ** 2
    )
)

print("\n========== SPATIAL LIGHTGBM RESULT ==========")
print(f"Validation RMSE : {spatial_lgb_rmse:.6f}")
print(f"Best iteration  : {lgb_spatial_model.best_iteration}")
print(f"Training time   : {training_time:.2f} minutes")

print("\n========== MODEL COMPARISON ==========")
print(f"Spatial XGBoost  : 3.863074")
print(f"Spatial LightGBM : {spatial_lgb_rmse:.6f}")

Validation shape: (5407429, 13)
Training shape  : (1000000, 12)
Validation shape: (5407429, 12)
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[940]	train's rmse: 3.94772	validation's rmse: 3.80715

========== SPATIAL LIGHTGBM RESULT ==========
Validation RMSE : 3.807145
Best iteration  : 940
Training time   : 2.31 minutes

========== MODEL COMPARISON ==========
Spatial XGBoost  : 3.863074
Spatial LightGBM : 3.807145


In [34]:
import os
import gc
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor

SPATIAL_VALIDATION_DIR = "/kaggle/working/taxi_spatial_validation"

TARGET = "fare_amount"

SPATIAL_FEATURES = [
    "passenger_count",
    "year",
    "month",
    "hour",
    "day_of_week",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "distance_km",
    "lat_difference",
    "lon_difference"
]

# ------------------------------------------------------------
# Load the SAME frozen validation set used by XGBoost and
# LightGBM.
# ------------------------------------------------------------

validation_files = sorted([
    os.path.join(SPATIAL_VALIDATION_DIR, f)
    for f in os.listdir(SPATIAL_VALIDATION_DIR)
    if f.endswith(".parquet")
])

validation_parts = []

for file_path in validation_files:
    part = pd.read_parquet(
        file_path,
        columns=SPATIAL_FEATURES + [TARGET]
    )
    validation_parts.append(part)

validation_spatial = pd.concat(
    validation_parts,
    ignore_index=True
)

del validation_parts
gc.collect()

assert len(validation_spatial) == 5_407_429

# ------------------------------------------------------------
# Prepare the identical feature matrices used by the other
# two spatial models.
# ------------------------------------------------------------

X_train_spatial = train_1m_spatial[SPATIAL_FEATURES]
y_train_spatial = train_1m_spatial[TARGET]

X_val_spatial = validation_spatial[SPATIAL_FEATURES]
y_val_spatial = validation_spatial[TARGET]

print("Training shape  :", X_train_spatial.shape)
print("Validation shape:", X_val_spatial.shape)

# ------------------------------------------------------------
# EXACT previous ExtraTrees configuration.
# We intentionally do not tune these parameters yet.
# ------------------------------------------------------------

et_spatial_model = ExtraTreesRegressor(
    n_estimators=100,
    max_depth=None,
    min_samples_leaf=5,
    max_features=1.0,
    n_jobs=-1,
    random_state=42
)

# ------------------------------------------------------------
# Train the model.
# ------------------------------------------------------------

start_time = time.time()

et_spatial_model.fit(
    X_train_spatial,
    y_train_spatial
)

training_time = (time.time() - start_time) / 60

# ------------------------------------------------------------
# Predict on the frozen validation set.
# ------------------------------------------------------------

val_predictions_spatial_et = et_spatial_model.predict(
    X_val_spatial
)

# ------------------------------------------------------------
# Calculate standard RMSE.
# Lower RMSE is better.
# ------------------------------------------------------------

spatial_et_rmse = np.sqrt(
    np.mean(
        (
            y_val_spatial.to_numpy()
            - val_predictions_spatial_et
        ) ** 2
    )
)

print("\n========== SPATIAL EXTRATREES RESULT ==========")
print(f"Validation RMSE : {spatial_et_rmse:.6f}")
print(f"Training time   : {training_time:.2f} minutes")

print("\n========== MODEL COMPARISON ==========")
print("Spatial XGBoost  : 3.863074")
print("Spatial LightGBM : 3.807145")
print(f"Spatial ExtraTrees: {spatial_et_rmse:.6f}")

Training shape  : (1000000, 12)
Validation shape: (5407429, 12)

========== SPATIAL EXTRATREES RESULT ==========
Validation RMSE : 3.880550
Training time   : 3.26 minutes

========== MODEL COMPARISON ==========
Spatial XGBoost  : 3.863074
Spatial LightGBM : 3.807145
Spatial ExtraTrees: 3.880550


In [35]:
import os
import gc
import time
import numpy as np
import pandas as pd
SPATIAL_TRAIN_POOL_DIR = "/kaggle/working/taxi_spatial_train_pool"
TARGET = "fare_amount"
SPATIAL_FEATURES = [
    "passenger_count",
    "year",
    "month",
    "hour",
    "day_of_week",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "distance_km",
    "lat_difference",
    "lon_difference"
]
TARGET_TRAIN_ROWS = 5_000_000
RANDOM_STATE = 42
# ------------------------------------------------------------
# Discover training pool files.
# ------------------------------------------------------------
train_pool_files = sorted([
    os.path.join(SPATIAL_TRAIN_POOL_DIR, f)
    for f in os.listdir(SPATIAL_TRAIN_POOL_DIR)
    if f.endswith(".parquet")
])
print(f"Training pool files: {len(train_pool_files)}")
# ------------------------------------------------------------
# Count rows in each part.
#
# We read ONLY the target column to minimise memory usage
# during the counting pass.
# ------------------------------------------------------------
part_sizes = []
for file_path in train_pool_files:
    n_rows = len(
        pd.read_parquet(
            file_path,
            columns=[TARGET]
        )
    )
    part_sizes.append(n_rows)
total_pool_rows = sum(part_sizes)
print(f"Training pool rows: {total_pool_rows:,}")
assert total_pool_rows == 48_655_705, (
    f"Unexpected training pool size: {total_pool_rows:,}"
)
# ------------------------------------------------------------
# Allocate 5M samples proportionally across parts.
#
# Uses the largest-remainder method (Hamilton's method) to
# guarantee exactly TARGET_TRAIN_ROWS after rounding.
# ------------------------------------------------------------
exact_allocations = (
    np.array(part_sizes, dtype=float)
    / total_pool_rows
    * TARGET_TRAIN_ROWS
)
sample_counts = np.floor(exact_allocations).astype(int)
remaining = TARGET_TRAIN_ROWS - sample_counts.sum()
if remaining > 0:
    fractional_parts = exact_allocations - sample_counts
    largest_remainders = np.argsort(fractional_parts)[-remaining:]
    sample_counts[largest_remainders] += 1
assert sample_counts.sum() == TARGET_TRAIN_ROWS, (
    f"Allocation error: {sample_counts.sum()} != {TARGET_TRAIN_ROWS}"
)
print(f"Rows to sample: {sample_counts.sum():,}")
# ------------------------------------------------------------
# Verify no part is asked for more rows than it contains.
# ------------------------------------------------------------
for i, (size, count) in enumerate(zip(part_sizes, sample_counts)):
    assert count <= size, (
        f"Part {i}: requested {count:,} but only {size:,} available"
    )
# ------------------------------------------------------------
# Sample from each part.
#
# We read only the required columns (12 features + target)
# to keep memory usage controlled during the sampling loop.
#
# Each part uses random_state = RANDOM_STATE + i for
# reproducibility and per-part independence.
# ------------------------------------------------------------
sample_parts = []
for i, (file_path, n_sample) in enumerate(
    zip(train_pool_files, sample_counts)
):
    if n_sample == 0:
        continue
    part = pd.read_parquet(
        file_path,
        columns=SPATIAL_FEATURES + [TARGET]
    )
    sampled_part = part.sample(
        n=n_sample,
        random_state=RANDOM_STATE + i
    )
    sample_parts.append(sampled_part)
    print(
        f"Part {i + 1:02d}/{len(train_pool_files)} | "
        f"available={len(part):,} | "
        f"sampled={n_sample:,}"
    )
    del part
    gc.collect()
# ------------------------------------------------------------
# Combine and shuffle.
# ------------------------------------------------------------
train_5m_spatial = pd.concat(
    sample_parts,
    ignore_index=True
)
del sample_parts
gc.collect()
train_5m_spatial = train_5m_spatial.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)
# ------------------------------------------------------------
# Integrity checks.
# ------------------------------------------------------------
assert len(train_5m_spatial) == TARGET_TRAIN_ROWS, (
    f"Row count mismatch: {len(train_5m_spatial):,} != {TARGET_TRAIN_ROWS:,}"
)
assert list(train_5m_spatial.columns) == SPATIAL_FEATURES + [TARGET], (
    "Column mismatch in training sample"
)
print("\n========== 5M SPATIAL SAMPLE COMPLETE ==========")
print(f"Shape : {train_5m_spatial.shape}")
print(f"Rows  : {len(train_5m_spatial):,}")
print(f"Memory: {train_5m_spatial.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
display(train_5m_spatial.head())


Training pool files: 56
Training pool rows: 48,655,705
Rows to sample: 5,000,000
Part 01/56 | available=878,402 | sampled=90,267
Part 02/56 | available=878,006 | sampled=90,226
Part 03/56 | available=877,359 | sampled=90,160
Part 04/56 | available=877,706 | sampled=90,196
Part 05/56 | available=878,265 | sampled=90,253
Part 06/56 | available=877,750 | sampled=90,200
Part 07/56 | available=878,329 | sampled=90,260
Part 08/56 | available=877,629 | sampled=90,188
Part 09/56 | available=877,751 | sampled=90,200
Part 10/56 | available=877,937 | sampled=90,219
Part 11/56 | available=877,620 | sampled=90,187
Part 12/56 | available=878,148 | sampled=90,241
Part 13/56 | available=878,114 | sampled=90,238
Part 14/56 | available=878,197 | sampled=90,246
Part 15/56 | available=878,342 | sampled=90,261
Part 16/56 | available=877,231 | sampled=90,147
Part 17/56 | available=877,439 | sampled=90,168
Part 18/56 | available=878,072 | sampled=90,233
Part 19/56 | available=878,289 | sampled=90,256
Part 20

,passenger_count,year,month,hour,day_of_week,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,distance_km,lat_difference,lon_difference,fare_amount
0,1,2013,7,22,2,-73.993515,40.751150,-73.949772,40.779588,4.854967,0.028438,0.043743,15.0
1,1,2014,7,14,2,-73.982132,40.774067,-73.954428,40.783942,2.578212,0.009875,0.027704,11.5
2,2,2009,12,23,3,-73.976442,40.751702,-73.981158,40.781313,3.316460,0.029611,0.004716,10.5
3,1,2012,4,7,0,-73.993345,40.752215,-73.979603,40.763067,1.672065,0.010852,0.013742,5.7
4,5,2009,4,13,1,-73.973262,40.792973,-73.973260,40.792982,0.001015,0.000009,0.000002,45.0


In [36]:
import os
import gc
import time
import numpy as np
import pandas as pd
import lightgbm as lgb
SPATIAL_VALIDATION_DIR = "/kaggle/working/taxi_spatial_validation"
SPATIAL_TRAIN_POOL_DIR = "/kaggle/working/taxi_spatial_train_pool"
TARGET = "fare_amount"
SPATIAL_FEATURES = [
    "passenger_count",
    "year",
    "month",
    "hour",
    "day_of_week",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "distance_km",
    "lat_difference",
    "lon_difference"
]
# ------------------------------------------------------------
# Assertion: training and validation are separate directories.
# ------------------------------------------------------------
assert SPATIAL_TRAIN_POOL_DIR != SPATIAL_VALIDATION_DIR, (
    "Training and validation directories must be different"
)
# ------------------------------------------------------------
# Load the frozen validation set.
# ------------------------------------------------------------
validation_files = sorted([
    os.path.join(SPATIAL_VALIDATION_DIR, f)
    for f in os.listdir(SPATIAL_VALIDATION_DIR)
    if f.endswith(".parquet")
])
validation_parts = []
for file_path in validation_files:
    part = pd.read_parquet(
        file_path,
        columns=SPATIAL_FEATURES + [TARGET]
    )
    validation_parts.append(part)
validation_spatial = pd.concat(
    validation_parts,
    ignore_index=True
)
del validation_parts
gc.collect()
assert len(validation_spatial) == 5_407_429, (
    f"Validation size mismatch: {len(validation_spatial):,}"
)
print("Validation shape:", validation_spatial.shape)
# ------------------------------------------------------------
# Prepare feature matrices.
# ------------------------------------------------------------
X_train = train_5m_spatial[SPATIAL_FEATURES]
y_train = train_5m_spatial[TARGET]
X_val = validation_spatial[SPATIAL_FEATURES]
y_val = validation_spatial[TARGET]
# ------------------------------------------------------------
# Assertion: target is NOT in the feature matrix.
# ------------------------------------------------------------
assert TARGET not in X_train.columns, (
    "Target column found in training features"
)
assert TARGET not in X_val.columns, (
    "Target column found in validation features"
)
# ------------------------------------------------------------
# Assertion: feature columns match exactly.
# ------------------------------------------------------------
assert list(X_train.columns) == SPATIAL_FEATURES
assert list(X_val.columns) == SPATIAL_FEATURES
print(f"Training shape  : {X_train.shape}")
print(f"Validation shape: {X_val.shape}")
# ------------------------------------------------------------
# EXACT same LightGBM parameters as the 1M experiment.
# Nothing is changed here.
# ------------------------------------------------------------
lgb_params = {
    "objective": "regression",
    "metric": "rmse",
    "num_leaves": 64,
    "max_depth": -1,
    "min_data_in_leaf": 100,
    "learning_rate": 0.05,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "seed": 42,
    "verbosity": -1,
    "n_jobs": -1
}
# ------------------------------------------------------------
# LightGBM dataset objects.
# ------------------------------------------------------------
lgb_train = lgb.Dataset(
    X_train,
    label=y_train
)
lgb_val = lgb.Dataset(
    X_val,
    label=y_val,
    reference=lgb_train
)
# ------------------------------------------------------------
# Free the pandas objects that are no longer needed.
#
# The LightGBM Datasets hold their own internal copies.
# We keep y_val for the manual RMSE calculation later.
# We keep train_5m_spatial in case it is needed downstream.
# ------------------------------------------------------------
del X_train, y_train, X_val
gc.collect()
# ------------------------------------------------------------
# Train with SAME early stopping as the 1M experiment.
# ------------------------------------------------------------
start_time = time.time()
lgb_5m_spatial_model = lgb.train(
    params=lgb_params,
    train_set=lgb_train,
    num_boost_round=2000,
    valid_sets=[
        lgb_train,
        lgb_val
    ],
    valid_names=[
        "train",
        "validation"
    ],
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            verbose=True
        ),
        lgb.log_evaluation(
            period=50
        )
    ]
)
training_time = (time.time() - start_time) / 60
# ------------------------------------------------------------
# Predict using the best iteration.
# ------------------------------------------------------------
val_predictions = lgb_5m_spatial_model.predict(
    validation_spatial[SPATIAL_FEATURES],
    num_iteration=lgb_5m_spatial_model.best_iteration
)
# ------------------------------------------------------------
# RMSE calculation — identical to the 1M experiment.
# ------------------------------------------------------------
val_rmse = np.sqrt(
    np.mean(
        (y_val.to_numpy() - val_predictions) ** 2
    )
)
print("\n========== 5M SPATIAL LIGHTGBM RESULT ==========")
print(f"Training rows   : {len(train_5m_spatial):,}")
print(f"Validation rows : {len(validation_spatial):,}")
print(f"Training memory : {train_5m_spatial.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Validation RMSE : {val_rmse:.6f}")
print(f"Best iteration  : {lgb_5m_spatial_model.best_iteration}")
print(f"Training time   : {training_time:.2f} minutes")
print("\n========== SCALING COMPARISON ==========")
print("1M Spatial LightGBM RMSE : 3.807145  (best iter: 940)")
print(f"5M Spatial LightGBM RMSE : {val_rmse:.6f}  (best iter: {lgb_5m_spatial_model.best_iteration})")
print(f"RMSE change              : {3.807145 - val_rmse:.6f}")


Validation shape: (5407429, 13)
Training shape  : (5000000, 12)
Validation shape: (5407429, 12)
Training until validation scores don't improve for 50 rounds
[50]	train's rmse: 4.20799	validation's rmse: 4.13235
[100]	train's rmse: 4.00746	validation's rmse: 3.93311
[150]	train's rmse: 3.94227	validation's rmse: 3.87401
[200]	train's rmse: 3.90069	validation's rmse: 3.83833
[250]	train's rmse: 3.86819	validation's rmse: 3.81185
[300]	train's rmse: 3.84484	validation's rmse: 3.7951
[350]	train's rmse: 3.82467	validation's rmse: 3.7814
[400]	train's rmse: 3.80786	validation's rmse: 3.77214
[450]	train's rmse: 3.7911	validation's rmse: 3.76185
[500]	train's rmse: 3.77725	validation's rmse: 3.75479
[550]	train's rmse: 3.76454	validation's rmse: 3.74919
[600]	train's rmse: 3.75343	validation's rmse: 3.74453
[650]	train's rmse: 3.74285	validation's rmse: 3.74066
[700]	train's rmse: 3.7329	validation's rmse: 3.73699
[750]	train's rmse: 3.72243	validation's rmse: 3.73319
[800]	train's rmse: 3.7

In [37]:
import os
import gc
import time
import numpy as np
import pandas as pd
import xgboost as xgb
SPATIAL_TRAIN_POOL_DIR = "/kaggle/working/taxi_spatial_train_pool"
SPATIAL_VALIDATION_DIR = "/kaggle/working/taxi_spatial_validation"
TARGET = "fare_amount"
SPATIAL_FEATURES = [
    "passenger_count",
    "year",
    "month",
    "hour",
    "day_of_week",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "distance_km",
    "lat_difference",
    "lon_difference"
]
# ------------------------------------------------------------
# Assertion: training and validation are separate directories.
# ------------------------------------------------------------
assert SPATIAL_TRAIN_POOL_DIR != SPATIAL_VALIDATION_DIR, (
    "Training and validation directories must be different"
)
# ------------------------------------------------------------
# Verify the existing 5M training sample.
# ------------------------------------------------------------
assert len(train_5m_spatial) == 5_000_000, (
    f"Training row count mismatch: {len(train_5m_spatial):,}"
)
assert list(train_5m_spatial.columns) == SPATIAL_FEATURES + [TARGET], (
    "Training column mismatch"
)
# ------------------------------------------------------------
# Load the frozen validation set.
# ------------------------------------------------------------
validation_files = sorted([
    os.path.join(SPATIAL_VALIDATION_DIR, f)
    for f in os.listdir(SPATIAL_VALIDATION_DIR)
    if f.endswith(".parquet")
])
validation_parts = []
for file_path in validation_files:
    part = pd.read_parquet(
        file_path,
        columns=SPATIAL_FEATURES + [TARGET]
    )
    validation_parts.append(part)
validation_spatial = pd.concat(
    validation_parts,
    ignore_index=True
)
del validation_parts
gc.collect()
assert len(validation_spatial) == 5_407_429, (
    f"Validation size mismatch: {len(validation_spatial):,}"
)
print("Validation shape:", validation_spatial.shape)
# ------------------------------------------------------------
# Prepare feature matrices.
# ------------------------------------------------------------
X_train = train_5m_spatial[SPATIAL_FEATURES]
y_train = train_5m_spatial[TARGET]
X_val = validation_spatial[SPATIAL_FEATURES]
y_val = validation_spatial[TARGET]
# ------------------------------------------------------------
# Assertions: target is not in the feature matrices.
# ------------------------------------------------------------
assert TARGET not in X_train.columns, (
    "Target column found in training features"
)
assert TARGET not in X_val.columns, (
    "Target column found in validation features"
)
assert list(X_train.columns) == SPATIAL_FEATURES
assert list(X_val.columns) == SPATIAL_FEATURES
print(f"Training shape  : {X_train.shape}")
print(f"Validation shape: {X_val.shape}")
# ------------------------------------------------------------
# XGBoost DMatrix construction.
# ------------------------------------------------------------
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)
# ------------------------------------------------------------
# Free intermediate pandas objects.
# DMatrix holds its own internal copy.
# y_val is retained for the manual RMSE calculation.
# ------------------------------------------------------------
del X_train, y_train, X_val
gc.collect()
# ------------------------------------------------------------
# EXACT same XGBoost parameters as the 1M experiment.
# Nothing is changed here.
# ------------------------------------------------------------
xgb_params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "max_depth": 8,
    "min_child_weight": 10,
    "eta": 0.10,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "tree_method": "hist",
    "nthread": -1,
    "seed": 42
}
# ------------------------------------------------------------
# Train with SAME early stopping as the 1M experiment.
# ------------------------------------------------------------
start_time = time.time()
xgb_5m_spatial_model = xgb.train(
    params=xgb_params,
    dtrain=dtrain,
    num_boost_round=500,
    evals=[
        (dtrain, "train"),
        (dval, "validation")
    ],
    early_stopping_rounds=30,
    verbose_eval=25
)
training_time = (time.time() - start_time) / 60
# ------------------------------------------------------------
# Predict using the best iteration.
#
# xgb.train() with early_stopping_rounds sets best_iteration
# on the Booster. Modern XGBoost predict() respects this
# automatically.
# ------------------------------------------------------------
val_predictions = xgb_5m_spatial_model.predict(dval)
# ------------------------------------------------------------
# RMSE calculation — identical formula to all prior experiments.
# ------------------------------------------------------------
val_rmse = np.sqrt(
    np.mean(
        (y_val.to_numpy() - val_predictions) ** 2
    )
)
# ------------------------------------------------------------
# Report.
# ------------------------------------------------------------
print("\n========== 5M SPATIAL XGBOOST RESULT ==========")
print(f"Training rows   : {len(train_5m_spatial):,}")
print(f"Validation rows : {len(validation_spatial):,}")
print(f"Training memory : {train_5m_spatial.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Validation RMSE : {val_rmse:.6f}")
print(f"Best iteration  : {xgb_5m_spatial_model.best_iteration}")
print(f"Training time   : {training_time:.2f} minutes")
print("\n========== SCALING COMPARISON ==========")
print("1M Spatial XGBoost RMSE : 3.863074  (best iter: 195)")
print(f"5M Spatial XGBoost RMSE : {val_rmse:.6f}  (best iter: {xgb_5m_spatial_model.best_iteration})")
print(f"RMSE change             : {3.863074 - val_rmse:.6f}")


Validation shape: (5407429, 13)
Training shape  : (5000000, 12)
Validation shape: (5407429, 12)
[0]	train-rmse:9.00829	validation-rmse:8.97598
[25]	train-rmse:4.10023	validation-rmse:4.04488
[50]	train-rmse:3.92180	validation-rmse:3.89183
[75]	train-rmse:3.84862	validation-rmse:3.85291
[100]	train-rmse:3.79205	validation-rmse:3.82011
[125]	train-rmse:3.74937	validation-rmse:3.80274
[150]	train-rmse:3.71221	validation-rmse:3.78738
[175]	train-rmse:3.68031	validation-rmse:3.77557
[200]	train-rmse:3.65192	validation-rmse:3.76863
[225]	train-rmse:3.62700	validation-rmse:3.76407
[250]	train-rmse:3.60740	validation-rmse:3.75992
[275]	train-rmse:3.58512	validation-rmse:3.75653
[300]	train-rmse:3.56514	validation-rmse:3.75454
[325]	train-rmse:3.54814	validation-rmse:3.75375
[350]	train-rmse:3.53356	validation-rmse:3.75300
[375]	train-rmse:3.51709	validation-rmse:3.75191
[400]	train-rmse:3.50045	validation-rmse:3.75205
[409]	train-rmse:3.49480	validation-rmse:3.75155

========== 5M SPATIAL XGBO

In [38]:
import os
import gc
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor
SPATIAL_VALIDATION_DIR = "/kaggle/working/taxi_spatial_validation"
TARGET = "fare_amount"
SPATIAL_FEATURES = [
    "passenger_count",
    "year",
    "month",
    "hour",
    "day_of_week",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "distance_km",
    "lat_difference",
    "lon_difference"
]
# ------------------------------------------------------------
# Verify the existing 5M training sample.
# ------------------------------------------------------------
assert len(train_5m_spatial) == 5_000_000, (
    f"Training row count mismatch: {len(train_5m_spatial):,}"
)
assert list(train_5m_spatial.columns) == SPATIAL_FEATURES + [TARGET], (
    "Training column mismatch"
)
# ------------------------------------------------------------
# Load the frozen validation set.
# ------------------------------------------------------------
validation_files = sorted([
    os.path.join(SPATIAL_VALIDATION_DIR, f)
    for f in os.listdir(SPATIAL_VALIDATION_DIR)
    if f.endswith(".parquet")
])
validation_parts = []
for file_path in validation_files:
    part = pd.read_parquet(
        file_path,
        columns=SPATIAL_FEATURES + [TARGET]
    )
    validation_parts.append(part)
validation_spatial = pd.concat(
    validation_parts,
    ignore_index=True
)
del validation_parts
gc.collect()
assert len(validation_spatial) == 5_407_429, (
    f"Validation size mismatch: {len(validation_spatial):,}"
)
print("Validation shape:", validation_spatial.shape)
# ------------------------------------------------------------
# Prepare feature matrices.
# ------------------------------------------------------------
X_train = train_5m_spatial[SPATIAL_FEATURES]
y_train = train_5m_spatial[TARGET]
X_val = validation_spatial[SPATIAL_FEATURES]
y_val = validation_spatial[TARGET]
# ------------------------------------------------------------
# Assertions: target is not in the feature matrices.
# ------------------------------------------------------------
assert TARGET not in X_train.columns, (
    "Target column found in training features"
)
assert TARGET not in X_val.columns, (
    "Target column found in validation features"
)
assert list(X_train.columns) == SPATIAL_FEATURES
assert list(X_val.columns) == SPATIAL_FEATURES
print(f"Training shape  : {X_train.shape}")
print(f"Validation shape: {X_val.shape}")
# ------------------------------------------------------------
# EXACT same ExtraTrees configuration as the 1M experiment.
# Nothing is changed here.
# ------------------------------------------------------------
et_5m_spatial_model = ExtraTreesRegressor(
    n_estimators=100,
    max_depth=None,
    min_samples_leaf=5,
    max_features=1.0,
    n_jobs=-1,
    random_state=42
)
# ------------------------------------------------------------
# Train.
#
# ExtraTrees with max_depth=None on 5M rows will build deep
# trees. This is memory-intensive but uses the same
# configuration as the 1M experiment.
# ------------------------------------------------------------
start_time = time.time()
et_5m_spatial_model.fit(X_train, y_train)
training_time = (time.time() - start_time) / 60
# ------------------------------------------------------------
# Predict on the frozen validation set.
# ------------------------------------------------------------
val_predictions = et_5m_spatial_model.predict(X_val)
# ------------------------------------------------------------
# RMSE calculation — identical formula to all prior experiments.
# ------------------------------------------------------------
val_rmse = np.sqrt(
    np.mean(
        (y_val.to_numpy() - val_predictions) ** 2
    )
)
# ------------------------------------------------------------
# Report.
# ------------------------------------------------------------
print("\n========== 5M SPATIAL EXTRATREES RESULT ==========")
print(f"Training rows   : {len(train_5m_spatial):,}")
print(f"Validation rows : {len(validation_spatial):,}")
print(f"Training memory : {train_5m_spatial.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Validation RMSE : {val_rmse:.6f}")
print(f"Training time   : {training_time:.2f} minutes")
print("\n========== SCALING COMPARISON ==========")
print("1M Spatial ExtraTrees RMSE : 3.880550")
print(f"5M Spatial ExtraTrees RMSE : {val_rmse:.6f}")
print(f"RMSE change                : {3.880550 - val_rmse:.6f}")
print("\n========== 5M THREE-MODEL COMPARISON ==========")
print("5M Spatial LightGBM    : 3.710197")
print("5M Spatial XGBoost     : 3.751553")
print(f"5M Spatial ExtraTrees  : {val_rmse:.6f}")

Validation shape: (5407429, 13)
Training shape  : (5000000, 12)
Validation shape: (5407429, 12)

========== 5M SPATIAL EXTRATREES RESULT ==========
Training rows   : 5,000,000
Validation rows : 5,407,429
Training memory : 257.49 MB
Validation RMSE : 3.744946
Training time   : 25.03 minutes

========== SCALING COMPARISON ==========
1M Spatial ExtraTrees RMSE : 3.880550
5M Spatial ExtraTrees RMSE : 3.744946
RMSE change                : 0.135604

========== 5M THREE-MODEL COMPARISON ==========
5M Spatial LightGBM    : 3.710197
5M Spatial XGBoost     : 3.751553
5M Spatial ExtraTrees  : 3.744946
